# DrowsyNet: Driver Drowsiness Detection

**Subject-independent drowsiness detection with facial landmarks, CNNs and temporal analysis, benchmarked across 30+ models and deployed as a Streamlit app (photo, video and live webcam).**

This notebook is the whole project in one file. It downloads the data, extracts features, trains and compares every model, picks the best one, and writes and launches the Streamlit app, which uses only that best model.

### What is new compared with the reference paper
Reference: C. Yashwanth & J. S. Kirar, *Driver's Drowsiness Detection*, IEEE TENCON 2019. They used dlib's 68 landmarks for eye/mouth aspect ratios and box heights, classified single frames with DT / NN / linear SVM / LDA, and trained and tested on the same person.

| Aspect | Reference paper | This project |
|---|---|---|
| Landmarks | dlib, 68 points | MediaPipe Face Landmarker: 478 points, **52 blendshapes**, head-pose matrix |
| Features | EAR, MAR, eye/mouth box height | EAR per eye, MAR, mouth opening, head pitch/yaw/roll, 51 blendshapes |
| Personal calibration | divides by the subject's average over the *whole* recording (EVD/AEVD), which includes the frames being classified | z-scores against the person's first alert frames only (first 20 % of alert frames here, first ~20 s in the app); those frames are never scored |
| Time | single frames | PERCLOS, blink duration, long closures, yawns, head nods (video / webcam) |
| Evaluation | same person in training and testing | **subject-disjoint** test set (people never seen in training) + 5-fold GroupKFold CV |
| Models | 4 classifiers | paper baseline + 14 classic ML on raw features + the same 14 with personal calibration + 8 CNN/ViT + stacking + late fusion (+ window and sequence models when video data is added) |
| Robustness | blurred frames broke 3 of 10 subjects | quality gate (blur / exposure), CLAHE low-light enhancement, CNN branch |
| Score | product of window averages, unbounded | smoothed, bounded 0–1 score mapped to the Karolinska Sleepiness Scale (KSS 1–9) |
| Explainability | none | reasons for every result, Grad-CAM heatmaps, feature importance |
| Product | offline experiment | Streamlit app: photo, video and live webcam with alarm |

### Pipeline
```
image / video / webcam
 └─ quality gate (blur, exposure) ─ CLAHE if dark
    └─ MediaPipe Face Landmarker → 478 landmarks + 52 blendshapes + head pose
       ├─ per-frame features (EAR, MAR, pose, blendshapes)
       └─ face crop (160 px)
          └─ SELECTED MODEL (winner of the benchmark) → drowsiness probability
             └─ temporal layer: calibration · PERCLOS · blinks · long closures · yawns · nods
                └─ smoothed score → alert / low vigilance / drowsy + KSS + reasons (+ alarm)
```

### How to run
1. Python 3.10–3.13. Run the install cell once.
2. **Run All**. Expensive steps (feature extraction, each model's training) are cached in `data/`, `models/` and `reports/`, so a re-run resumes where it stopped. Set `FORCE_RETRAIN = True` to retrain everything.
3. The last section starts the Streamlit app at http://localhost:8501.

## 0. Setup

In [ ]:
# Run once. CPU-only PyTorch is enough; on Colab / a GPU machine plain `pip install torch torchvision` is fine.
import importlib.util, subprocess, sys
needed = {"torch": "torch torchvision --index-url https://download.pytorch.org/whl/cpu",
          "mediapipe": "mediapipe", "cv2": "opencv-python", "sklearn": "scikit-learn", "xgboost": "xgboost",
          "lightgbm": "lightgbm", "catboost": "catboost", "timm": "timm", "kagglehub": "kagglehub",
          "streamlit": "streamlit", "plotly": "plotly", "seaborn": "seaborn", "tqdm": "tqdm"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
for pkg in missing:
    print("installing", pkg)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkg.split()])
print("all packages available" if not missing else "installed: " + ", ".join(missing))

In [ ]:
# ---- Run settings -----------------------------------------------------------------------
FORCE_RETRAIN = False      # True = ignore cached results and retrain every model
RUN_CV = True              # 5-fold subject-grouped cross-validation for the classic models
CNN_EPOCHS = 3             # CPU budget; raise on a GPU
CNN_TRAIN_STRIDE = 4       # use every 4th training frame (DDD frames are near-duplicates)
TRAIN_HEAVY_CNN = True     # ResNet-50, VGG-16, DeiT-Tiny (slow on CPU -> reduced budget below)
HEAVY_TRAIN_STRIDE = 8
HEAVY_EPOCHS = 2
USE_RLDD = False           # set True after placing UTA-RLDD videos in data/raw/rldd (Groups B and D)
EXTRACT_WORKERS = 3        # parallel MediaPipe workers for feature extraction

import os, json, time, glob, shutil, warnings
from pathlib import Path
import cv2
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40, "display.width", 200)
PROJECT = Path.cwd()
print("project folder:", PROJECT)

## 1. Data: Driver Drowsiness Dataset (DDD)

[Kaggle: Driver Drowsiness Dataset (DDD)](https://www.kaggle.com/datasets/ismailnasri20/driver-drowsiness-dataset-ddd) has 41,793 face images (227×227) labelled **Drowsy** or **Non Drowsy**. They were cut from the UTA Real-Life Drowsiness Dataset videos. The filename's letter prefix identifies the participant (`A0001.png`, `a0002.png` = participant A), which is what makes a **subject-disjoint** evaluation possible.

> DDD frames are consecutive video frames, so neighbouring images are near-duplicates. A random train/test split puts the same person and moment in both sets and gives the inflated ~99 % accuracies common in online notebooks. Here every reported number comes from people the model has never seen.

In [ ]:
import kagglehub
DDD_ROOT = Path(kagglehub.dataset_download("ismailnasri20/driver-drowsiness-dataset-ddd"))
print("DDD downloaded to:", DDD_ROOT)

## 2. The shared pipeline: `drowsy_core.py`

The pipeline (quality gate → landmarks → features → temporal layer → scoring → model loading → inference engine) is written below in several cells using `%%writefile`. The notebook imports it, **and the Streamlit app imports the same file**, so training and the live app use exactly the same feature code.

### 2.1 Configuration

In [ ]:
%%writefile drowsy_core.py
"""DrowsyNet core pipeline (generated by DrowsyNet.ipynb — edit the notebook, not this file)."""
from __future__ import annotations

import base64, json, math, os, threading, time, urllib.request
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path

os.environ.setdefault("GLOG_minloglevel", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import cv2
import numpy as np

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = ROOT / "models"
CANDIDATES_DIR = MODELS_DIR / "candidates"
BEST_DIR = MODELS_DIR / "best"
REPORTS_DIR = ROOT / "reports"
RESULTS_DIR = REPORTS_DIR / "results"
PREDICTIONS_DIR = REPORTS_DIR / "predictions"
FIGURES_DIR = REPORTS_DIR / "figures"
ASSETS_DIR = ROOT / "assets"
FACE_LANDMARKER_PATH = ASSETS_DIR / "face_landmarker.task"
FACE_LANDMARKER_URL = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
                       "face_landmarker/float16/1/face_landmarker.task")
SEED = 42

MAX_MODEL_LATENCY_MS = 30.0     # selection rule: model-only latency per frame on CPU
F1_TIE_TOLERANCE = 0.005
TARGET_FPS = 10                 # frames analysed per second (video / webcam)
CALIBRATION_SECONDS = 20        # personal baseline at the start of a session
PERCLOS_WINDOW_S = 30
EVENT_WINDOW_S = 60
PROB_WINDOW_S = 3

for _d in (PROCESSED_DIR, SPLITS_DIR, CANDIDATES_DIR, BEST_DIR, RESULTS_DIR, PREDICTIONS_DIR, FIGURES_DIR, ASSETS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

### 2.2 Frame-quality gate
Every frame gets a blur score (variance of the Laplacian) and a brightness score. Unusable frames are skipped by the temporal layer, and dark frames get CLAHE contrast enhancement. This targets the failure the paper reported: blurred frames ruined 3 of its 10 subjects.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ quality gate
BLUR_MIN, DARK_MAX, BRIGHT_MIN, BRIGHT_MAX = 25.0, 60.0, 25.0, 235.0
_clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))


def quality_check(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    bright = float(gray.mean())
    q = {"blur": blur, "brightness": bright, "usable": blur >= BLUR_MIN and BRIGHT_MIN <= bright <= BRIGHT_MAX}
    if bright < DARK_MAX:                                   # low light -> CLAHE on the L channel
        l, a, b = cv2.split(cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB))
        bgr = cv2.cvtColor(cv2.merge((_clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)
    return bgr, q

### 2.3 Face landmarks (MediaPipe Face Landmarker)
For each face it returns 478 3-D landmarks, 52 blendshape scores (for example `eyeBlinkLeft`, `eyeSquintRight`, `jawOpen`) and a 4×4 head-pose matrix. DDD images are tight face crops, which the face detector can miss, so detection is retried once with a padded border.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ landmarks
def ensure_landmarker_model():
    if not FACE_LANDMARKER_PATH.exists():
        print("downloading MediaPipe face landmarker model ...")
        urllib.request.urlretrieve(FACE_LANDMARKER_URL, FACE_LANDMARKER_PATH)
    return FACE_LANDMARKER_PATH


@dataclass
class FaceResult:
    points: np.ndarray          # (478, 3) pixel coordinates
    blendshapes: dict
    matrix: np.ndarray | None   # 4x4 facial transformation matrix
    bbox: tuple                 # x1, y1, x2, y2
    image_size: tuple           # (w, h)


class FaceAnalyzer:
    def __init__(self, min_confidence=0.3):
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
        self._mp = mp
        opts = vision.FaceLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_buffer=ensure_landmarker_model().read_bytes()),
            running_mode=vision.RunningMode.IMAGE, num_faces=1,
            min_face_detection_confidence=min_confidence, min_face_presence_confidence=min_confidence,
            output_face_blendshapes=True, output_facial_transformation_matrixes=True)
        self._lm = vision.FaceLandmarker.create_from_options(opts)
        self._lock = threading.Lock()

    def _raw(self, bgr):
        img = self._mp.Image(image_format=self._mp.ImageFormat.SRGB,
                             data=np.ascontiguousarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
        with self._lock:
            return self._lm.detect(img)

    def detect(self, bgr):
        h, w = bgr.shape[:2]
        res, off, (ph, pw) = self._raw(bgr), 0, (h, w)
        if not res.face_landmarks:                           # tight crops: retry with a border
            off = int(0.35 * max(h, w))
            padded = cv2.copyMakeBorder(bgr, off, off, off, off, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            res, (ph, pw) = self._raw(padded), padded.shape[:2]
        if not res.face_landmarks:
            return None
        pts = np.array([[p.x * pw - off, p.y * ph - off, p.z * pw] for p in res.face_landmarks[0]], np.float32)
        blend = {c.category_name: float(c.score) for c in res.face_blendshapes[0]} if res.face_blendshapes else {}
        mat = np.asarray(res.facial_transformation_matrixes[0], np.float32) if res.facial_transformation_matrixes else None
        x1, y1 = np.clip(pts[:, 0].min(), 0, w - 1), np.clip(pts[:, 1].min(), 0, h - 1)
        x2, y2 = np.clip(pts[:, 0].max(), 0, w - 1), np.clip(pts[:, 1].max(), 0, h - 1)
        return FaceResult(pts, blend, mat, (int(x1), int(y1), int(x2), int(y2)), (w, h))

### 2.4 Per-frame features
* **EAR** (eye aspect ratio, Soukupová & Čech 2016): $EAR=\frac{\lVert p_2-p_6\rVert+\lVert p_3-p_5\rVert}{2\lVert p_1-p_4\rVert}$, computed for each eye. It drops towards 0 as the eye closes.
* **MAR** (mouth aspect ratio): three vertical lip distances divided by 3× the mouth width. It rises when yawning. (The paper's Eq. 1 has a sign error, "−" instead of "+", and misplaced brackets.)
* **Head pose**: pitch / yaw / roll from the transformation matrix. Nodding off shows up as pitch changes.
* **Blendshapes**: 51 learned facial-action scores (`_neutral` is dropped).
* **Paper features**: EAR + MAR only, used to reproduce the reference setup.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ features
RIGHT_EYE, LEFT_EYE = [33, 160, 158, 133, 153, 144], [362, 385, 387, 263, 373, 380]
MOUTH_CORNERS, MOUTH_VERTICAL = (61, 291), [(81, 178), (13, 14), (311, 402)]
FACE_TOP, FACE_BOTTOM = 10, 152
CONTOURS = {
    "right_eye": [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246],
    "left_eye": [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398],
    "mouth": [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
}
BLENDSHAPE_NAMES = [
    "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight", "cheekPuff",
    "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight", "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
    "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight", "jawForward", "jawLeft", "jawOpen", "jawRight",
    "mouthClose", "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight", "mouthFunnel",
    "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft", "mouthPressRight", "mouthPucker",
    "mouthRight", "mouthRollLower", "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper", "mouthSmileLeft",
    "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft", "mouthUpperUpRight",
    "noseSneerLeft", "noseSneerRight"]
GEOM_FEATURES = ["ear_left", "ear_right", "ear_mean", "ear_diff", "mar", "mouth_open", "pitch", "yaw", "roll", "face_scale"]
BLEND_FEATURES = [f"bs_{n}" for n in BLENDSHAPE_NAMES]
FRAME_FEATURES = GEOM_FEATURES + BLEND_FEATURES
PAPER_FEATURES = ["ear_mean", "mar"]


def _d(a, b):
    return float(np.linalg.norm(a - b))


def eye_aspect_ratio(pts, idx):
    p = pts[idx, :2]
    return (_d(p[1], p[5]) + _d(p[2], p[4])) / (2.0 * _d(p[0], p[3]) + 1e-6)


def mouth_aspect_ratio(pts):
    p = pts[:, :2]
    return sum(_d(p[a], p[b]) for a, b in MOUTH_VERTICAL) / (3.0 * _d(p[MOUTH_CORNERS[0]], p[MOUTH_CORNERS[1]]) + 1e-6)


def head_pose(m):
    if m is None:
        return 0.0, 0.0, 0.0
    r = m[:3, :3]
    sy = math.sqrt(r[0, 0] ** 2 + r[1, 0] ** 2)
    pitch = math.atan2(r[2, 1], r[2, 2]) if sy > 1e-6 else math.atan2(-r[1, 2], r[1, 1])
    yaw = math.atan2(-r[2, 0], sy)
    roll = math.atan2(r[1, 0], r[0, 0]) if sy > 1e-6 else 0.0
    return math.degrees(pitch), math.degrees(yaw), math.degrees(roll)


def frame_features(face: FaceResult) -> dict:
    pts = face.points
    er, el = eye_aspect_ratio(pts, RIGHT_EYE), eye_aspect_ratio(pts, LEFT_EYE)
    face_h = _d(pts[FACE_TOP, :2], pts[FACE_BOTTOM, :2]) + 1e-6
    pitch, yaw, roll = head_pose(face.matrix)
    f = {"ear_left": el, "ear_right": er, "ear_mean": (el + er) / 2, "ear_diff": abs(el - er),
         "mar": mouth_aspect_ratio(pts), "mouth_open": _d(pts[13, :2], pts[14, :2]) / face_h,
         "pitch": pitch, "yaw": yaw, "roll": roll, "face_scale": face_h / max(face.image_size[1], 1)}
    for n in BLENDSHAPE_NAMES:
        f[f"bs_{n}"] = face.blendshapes.get(n, 0.0)
    return f


def overlay_points(face: FaceResult) -> dict:
    w, h = face.image_size
    return {k: [[float(face.points[i, 0] / w), float(face.points[i, 1] / h)] for i in idx] for k, idx in CONTOURS.items()}


def draw_overlay(bgr, overlay, color=(238, 211, 34), thickness=2):
    h, w = bgr.shape[:2]
    out = bgr.copy()
    for pts in (overlay or {}).values():
        arr = np.array([[x * w, y * h] for x, y in pts], np.int32)
        cv2.polylines(out, [arr], True, color, thickness, cv2.LINE_AA)
    return out


def crop_face(bgr, face: FaceResult, size=160, margin=0.12):
    h, w = bgr.shape[:2]
    x1, y1, x2, y2 = face.bbox
    cx, cy, half = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * (1 + 2 * margin) / 2
    a, b, c, d = (int(round(v)) for v in (cx - half, cy - half, cx + half, cy + half))
    pad = max(0, -a, -b, c - w, d - h)
    if pad:
        bgr = cv2.copyMakeBorder(bgr, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
        a, b, c, d = a + pad, b + pad, c + pad, d + pad
    crop = bgr[b:d, a:c]
    return cv2.resize(crop if crop.size else bgr, (size, size), interpolation=cv2.INTER_AREA)

### 2.5 Temporal layer: personal calibration and events
For video and webcam, a `SessionState` learns **this driver's own open-eye EAR and head pitch** during the first ~20 s. Thresholds are then relative to that baseline, not one fixed number for everyone. From the stream it tracks:
* **PERCLOS**: percentage of time the eyes were closed over the last 30 s (a standard drowsiness measure)
* **blinks** and **long closures** (≥ 0.5 s, a sign of micro-sleeps)
* **yawns**: `jawOpen` held for ≥ 1 s
* **head nods**: pitch moving > 12° away from the driver's baseline

`window_features` turns a fixed window of frames into aggregate features for the Group B models.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ temporal layer
EYE_CLOSED_RATIO, EYE_CLOSED_ABS, BLINK_MAX_S = 0.70, 0.18, 0.5
YAWN_JAW, YAWN_MIN_S, NOD_PITCH_DEG = 0.55, 1.0, 12.0


@dataclass
class Baseline:
    ear: float = 0.0
    mar: float = 0.0
    pitch: float = 0.0
    ready: bool = False


@dataclass
class SessionState:
    fps: float = TARGET_FPS
    calibration_s: float = CALIBRATION_SECONDS
    baseline: Baseline = field(default_factory=Baseline)
    _calib: list = field(default_factory=list)
    _frames: deque = field(default_factory=deque)     # (t, closed, prob, usable)
    _events: deque = field(default_factory=deque)     # (t, kind, duration)
    _closed_since: float | None = None
    _yawn_since: float | None = None
    _nod_active: bool = False
    _start_t: float | None = None
    blinks_total: int = 0
    yawns_total: int = 0
    nods_total: int = 0
    long_closures_total: int = 0

    def is_closed(self, f):
        thr = EYE_CLOSED_RATIO * self.baseline.ear if self.baseline.ready else EYE_CLOSED_ABS
        return f["ear_mean"] < thr or (f["bs_eyeBlinkLeft"] + f["bs_eyeBlinkRight"]) / 2 > 0.55

    def _calibrate(self, t, f):
        if self.baseline.ready:
            return
        self._start_t = t if self._start_t is None else self._start_t
        if f["bs_eyeBlinkLeft"] < 0.4 and f["bs_eyeBlinkRight"] < 0.4:
            self._calib.append((f["ear_mean"], f["mar"], f["pitch"]))
        if t - self._start_t >= self.calibration_s and len(self._calib) >= 10:
            a = np.array(self._calib)
            self.baseline = Baseline(float(np.median(a[:, 0])), float(np.median(a[:, 1])), float(np.median(a[:, 2])), True)

    def update(self, t, f, prob, usable=True):
        horizon = max(PERCLOS_WINDOW_S, EVENT_WINDOW_S)
        while self._frames and t - self._frames[0][0] > horizon:
            self._frames.popleft()
        while self._events and t - self._events[0][0] > EVENT_WINDOW_S:
            self._events.popleft()
        if f is None or not usable:
            self._frames.append((t, None, prob, False))
            return self.summary(t)
        self._calibrate(t, f)
        closed = self.is_closed(f)
        if closed and self._closed_since is None:
            self._closed_since = t
        elif not closed and self._closed_since is not None:
            dur, self._closed_since = t - self._closed_since, None
            kind = "long_closure" if dur >= BLINK_MAX_S else "blink"
            self._events.append((t, kind, dur))
            if kind == "blink":
                self.blinks_total += 1
            else:
                self.long_closures_total += 1
        yawning = f["bs_jawOpen"] > YAWN_JAW
        if yawning and self._yawn_since is None:
            self._yawn_since = t
        elif not yawning and self._yawn_since is not None:
            if t - self._yawn_since >= YAWN_MIN_S:
                self._events.append((t, "yawn", t - self._yawn_since))
                self.yawns_total += 1
            self._yawn_since = None
        if self.baseline.ready:
            nodding = abs(f["pitch"] - self.baseline.pitch) > NOD_PITCH_DEG
            if nodding and not self._nod_active:
                self._events.append((t, "nod", 0.0))
                self.nods_total += 1
            self._nod_active = nodding
        self._frames.append((t, closed, prob, True))
        return self.summary(t)

    def summary(self, t):
        recent = [x for x in self._frames if t - x[0] <= PERCLOS_WINDOW_S]
        good = [x for x in recent if x[3]]
        probs = [x[2] for x in good if x[2] is not None and t - x[0] <= PROB_WINDOW_S]
        ev = list(self._events)
        blinks = [e for e in ev if e[1] == "blink"]
        prog = 1.0 if self.baseline.ready else (0.0 if self._start_t is None else min((t - self._start_t) / self.calibration_s, 0.99))
        return {"perclos": float(np.mean([x[1] for x in good])) if good else 0.0,
                "prob_window": float(np.mean(probs)) if probs else None,
                "blinks_per_min": len(blinks) * 60.0 / EVENT_WINDOW_S,
                "mean_blink_s": float(np.mean([e[2] for e in blinks])) if blinks else 0.0,
                "long_closures": sum(e[1] == "long_closure" for e in ev),
                "current_closure_s": (t - self._closed_since) if self._closed_since is not None else 0.0,
                "yawns": sum(e[1] == "yawn" for e in ev), "nods": sum(e[1] == "nod" for e in ev),
                "usable_ratio": len(good) / len(recent) if recent else 0.0,
                "calibrated": self.baseline.ready, "calibration_progress": float(prog)}


WINDOW_BASE = ["ear_mean", "mar", "mouth_open", "pitch", "yaw", "roll", "bs_eyeBlinkLeft", "bs_eyeBlinkRight",
               "bs_eyeSquintLeft", "bs_eyeSquintRight", "bs_jawOpen", "bs_eyeLookDownLeft", "bs_eyeLookDownRight",
               "bs_browInnerUp"]


def _runs(mask):
    runs, n = [], 0
    for v in mask:
        if v:
            n += 1
        elif n:
            runs.append(n)
            n = 0
    return runs + ([n] if n else [])


def window_features(win: dict, fps: float) -> dict:
    out = {}
    for c in WINDOW_BASE:
        x = np.asarray(win[c], np.float32)
        out.update({f"{c}_mean": float(x.mean()), f"{c}_std": float(x.std()), f"{c}_min": float(x.min()), f"{c}_max": float(x.max())})
    ear = np.asarray(win["ear_mean"], np.float32)
    closed = (ear < EYE_CLOSED_ABS) | ((np.asarray(win["bs_eyeBlinkLeft"]) + np.asarray(win["bs_eyeBlinkRight"])) / 2 > 0.55)
    runs = np.array(_runs(closed), np.float32) / fps
    yawn = np.array(_runs(np.asarray(win["bs_jawOpen"]) > YAWN_JAW), np.float32) / fps
    secs = len(ear) / fps
    out.update({"perclos": float(closed.mean()),
                "blink_rate": float((runs < BLINK_MAX_S).sum() * 60 / secs) if len(runs) else 0.0,
                "blink_dur_mean": float(runs.mean()) if len(runs) else 0.0,
                "closure_max_s": float(runs.max()) if len(runs) else 0.0,
                "long_closures": float((runs >= BLINK_MAX_S).sum()) if len(runs) else 0.0,
                "yawns": float((yawn >= YAWN_MIN_S).sum()) if len(yawn) else 0.0,
                "pitch_range": float(np.ptp(np.asarray(win["pitch"], np.float32)))})
    return out

### 2.6 Scoring: from evidence to a drowsiness score
`score = 0.6 × (model probability averaged over 3 s) + 0.4 × temporal evidence (PERCLOS, closures, yawns, nods)`, then smoothed with an asymmetric moving average that rises fast and falls slowly, so the alarm fires quickly but doesn't flicker. Score bands: < 0.40 **alert**, 0.40–0.65 **low vigilance**, ≥ 0.65 **drowsy**. The score is also mapped linearly to the KSS (1–9).

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ scoring
ALERT_MAX, DROWSY_MIN = 0.40, 0.65
KSS_LABELS = {1: "Extremely alert", 2: "Very alert", 3: "Alert", 4: "Rather alert", 5: "Neither alert nor sleepy",
              6: "Some signs of sleepiness", 7: "Sleepy, no effort to stay awake",
              8: "Sleepy, some effort to stay awake", 9: "Very sleepy, fighting sleep"}


class Smoother:
    def __init__(self, rise=0.35, decay=0.12):
        self.rise, self.decay, self.value = rise, decay, None

    def __call__(self, x):
        if self.value is None:
            self.value = x
        else:
            a = self.rise if x > self.value else self.decay
            self.value = a * x + (1 - a) * self.value
        return self.value


def rule_score(s):
    return min(1.0, 0.40 * min(s["perclos"] / 0.35, 1) + 0.25 * max(min(s["current_closure_s"] / 1.5, 1), min(s["long_closures"] / 3, 1))
               + 0.20 * min(s["yawns"] / 3, 1) + 0.15 * min(s["nods"] / 3, 1))


def combine(prob_window, s, w=0.6):
    return rule_score(s) if prob_window is None else w * prob_window + (1 - w) * rule_score(s)


def state_of(score):
    return "alert" if score < ALERT_MAX else ("low_vigilance" if score < DROWSY_MIN else "drowsy")


def kss_of(score):
    return int(min(9, max(1, round(1 + score * 8))))


def reasons(s, prob_window, snap=None):
    out = []
    if prob_window is not None and prob_window >= 0.6:
        out.append(f"Model confidence for a drowsy face: {prob_window:.0%}")
    if s.get("perclos", 0) >= 0.15:
        out.append(f"Eyes closed {s['perclos']:.0%} of the last 30 s (PERCLOS)")
    if s.get("current_closure_s", 0) >= 0.5:
        out.append(f"Eyes closed right now for {s['current_closure_s']:.1f} s")
    if s.get("long_closures", 0):
        out.append(f"{s['long_closures']} long eye closure(s) in the last minute")
    if s.get("yawns", 0):
        out.append(f"{s['yawns']} yawn(s) in the last minute")
    if s.get("nods", 0):
        out.append(f"{s['nods']} head nod(s) in the last minute")
    if snap:
        if snap.get("eyes_closed"):
            out.append("Eyes appear closed in this image")
        if snap.get("yawning"):
            out.append("Mouth wide open (yawn)")
        if abs(snap.get("pitch", 0)) > 20:
            out.append("Head tilted strongly")
    return out

### 2.7 Model architectures
* **Classic ML** (paper baseline, Groups A and B, stacking): every model is a scikit-learn `Pipeline(StandardScaler → classifier)`.
* **Group C image models**: a custom 4-block CNN trained from scratch, plus ImageNet-pretrained backbones from `timm` (MobileNetV3, EfficientNet, ResNet, VGG) and a Vision Transformer (DeiT-Tiny), all fine-tuned on face crops.
* **Group D sequence models**: 1D-CNN, TCN (causal dilated convolutions), LSTM, BiLSTM, GRU and a Transformer encoder over windows of per-frame features.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ model architectures
def classic_estimator(kind: str):
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.svm import SVC, LinearSVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier
    xgb = lambda: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.8,
                                eval_metric="logloss", n_jobs=-1, random_state=SEED)
    lgbm = lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.9, subsample_freq=1,
                                  colsample_bytree=0.8, verbose=-1, random_state=SEED)
    rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, class_weight="balanced_subsample", random_state=SEED)
    table = {
        "logreg": lambda: LogisticRegression(max_iter=2000, class_weight="balanced"),
        "knn": lambda: KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),
        "gnb": lambda: GaussianNB(),
        "lda": lambda: LinearDiscriminantAnalysis(),
        "dtree": lambda: DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=SEED),
        "linsvm": lambda: CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced", max_iter=5000), cv=3),
        "rbfsvm": lambda: SVC(C=3.0, gamma="scale", probability=True, class_weight="balanced", random_state=SEED),
        "rf": rf,
        "extratrees": lambda: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, class_weight="balanced", random_state=SEED),
        "hgb": lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, random_state=SEED),
        "xgb": xgb,
        "lgbm": lgbm,
        "catboost": lambda: CatBoostClassifier(iterations=400, depth=6, learning_rate=0.08, verbose=0, random_seed=SEED,
                                               thread_count=-1, allow_writing_files=False),
        "mlp": lambda: MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, early_stopping=True, max_iter=300, random_state=SEED),
        "stacking": lambda: StackingClassifier([("rf", rf()), ("xgb", xgb()), ("lgbm", lgbm())],
                                               final_estimator=LogisticRegression(max_iter=1000), cv=3, stack_method="predict_proba"),
    }
    return table[kind]()


def classic_pipeline(model_id: str):
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    return Pipeline([("scale", StandardScaler()), ("clf", classic_estimator(model_id.split("_", 1)[1]))])


IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def build_cnn(arch: str, pretrained=True, img=128):
    import torch.nn as nn
    if arch == "custom":
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        return nn.Sequential(nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 192)),
                             nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(192, 2)))
    import timm
    kw = {"img_size": img} if arch.startswith("deit") else {}
    return timm.create_model(arch, pretrained=pretrained, num_classes=2, **kw)


def gradcam_layer(model, arch):
    if arch == "custom":
        return model[0][-1]
    if arch.startswith("deit"):
        return None
    if arch.startswith("resnet"):
        return model.layer4
    if arch.startswith("vgg"):
        return model.features
    return model.blocks[-1] if hasattr(model, "blocks") else None


def cnn_preprocess(bgr_list, img):
    import torch
    arr = np.stack([cv2.resize(cv2.cvtColor(b, cv2.COLOR_BGR2RGB), (img, img), interpolation=cv2.INTER_AREA) for b in bgr_list]).astype(np.float32) / 255.0
    arr = (arr - np.array(IMAGENET_MEAN, np.float32)) / np.array(IMAGENET_STD, np.float32)
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()


class CropDataset:
    """Face-crop dataset for CNN training (in this module so DataLoader workers can import it on Windows)."""

    def __init__(self, paths, labels, img, train):
        from torchvision import transforms as T
        self.paths, self.labels = list(paths), [int(x) for x in labels]
        aug = [T.RandomResizedCrop(img, scale=(0.8, 1.0)), T.RandomHorizontalFlip(),
               T.RandomApply([T.ColorJitter(0.4, 0.4, 0.3, 0.05)], p=0.8),
               T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3), T.RandomGrayscale(p=0.15), T.RandomRotation(10)]
        self.tf = T.Compose([T.ToPILImage(), *(aug if train else [T.Resize((img, img))]), T.ToTensor(),
                             T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(cv2.cvtColor(cv2.imread(self.paths[i]), cv2.COLOR_BGR2RGB)), self.labels[i]


def build_sequence(arch: str, n_feat: int):
    import torch
    import torch.nn as nn

    class RNN(nn.Module):
        def __init__(self, cell, bi):
            super().__init__()
            self.rnn = {"lstm": nn.LSTM, "gru": nn.GRU}[cell](n_feat, 64, 2, batch_first=True, dropout=0.2, bidirectional=bi)
            self.head = nn.Sequential(nn.LayerNorm(128 if bi else 64), nn.Dropout(0.2), nn.Linear(128 if bi else 64, 2))

        def forward(self, x):
            return self.head(self.rnn(x)[0].mean(1))

    class Conv1D(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Conv1d(n_feat, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
                                     nn.Conv1d(64, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
                                     nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
                                     nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, 2))

        def forward(self, x):
            return self.net(x.transpose(1, 2))

    class TCN(nn.Module):
        def __init__(self, ch=64, levels=4):
            super().__init__()
            self.convs, self.res, self.pads = nn.ModuleList(), nn.ModuleList(), []
            cin = n_feat
            for i in range(levels):
                d = 2 ** i
                self.convs.append(nn.Conv1d(cin, ch, 3, padding=2 * d, dilation=d))
                self.res.append(nn.Conv1d(cin, ch, 1) if cin != ch else nn.Identity())
                self.pads.append(2 * d)
                cin = ch
            self.head = nn.Linear(ch, 2)

        def forward(self, x):
            h = x.transpose(1, 2)
            for conv, res, p in zip(self.convs, self.res, self.pads):
                h = torch.relu(conv(h)[..., :-p] + res(h))     # causal: drop the look-ahead part
            return self.head(h[..., -1])

    class Transformer(nn.Module):
        def __init__(self, d=64):
            super().__init__()
            self.inp = nn.Linear(n_feat, d)
            self.pos = nn.Parameter(torch.zeros(1, 512, d))
            self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 128, 0.2, batch_first=True), 2)
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2))

        def forward(self, x):
            return self.head(self.enc(self.inp(x) + self.pos[:, : x.size(1)]).mean(1))

    return {"lstm": lambda: RNN("lstm", False), "bilstm": lambda: RNN("lstm", True), "gru": lambda: RNN("gru", False),
            "conv1d": Conv1D, "tcn": TCN, "transformer": Transformer}[arch]()

### 2.8 Loading a trained model and the inference engine
Every trained model is saved under `models/candidates/<id>/` with a `meta.json` that records its kind. `load_model` works for every kind, so the app can serve whichever model won.

`DrowsinessEngine` runs the full pipeline:
* **photo**: a snapshot check (model probability + eyes closed / yawn cues) with Grad-CAM. A single photo has no time information.
* **video / webcam**: per-frame analysis plus the temporal layer, which gives the smoothed score, events and alarm.

In [ ]:
%%writefile -a drowsy_core.py


# ============================================================================ personal calibration
Z_FEATURES = [f"z_{c}" for c in FRAME_FEATURES]


def calibration_stats(feature_rows):
    """Personal baseline from frames recorded while the driver is alert: per-feature median and std."""
    a = np.array([[r[c] for c in FRAME_FEATURES] for r in feature_rows], np.float64)
    return np.median(a, 0), a.std(0) + 1e-3


def z_features(feats, baseline):
    med, sd = baseline
    x = np.array([feats.get(c, 0.0) for c in FRAME_FEATURES])
    return dict(zip(Z_FEATURES, np.clip((x - med) / sd, -50, 50)))


# ============================================================================ trained-model loading
class ClassicFrameModel:
    per_frame = True

    def __init__(self, d, meta):
        import joblib
        self.model, self.features, self.meta = joblib.load(d / "model.joblib"), meta["features"], meta
        self.needs_calibration = bool(meta.get("calibrated"))
        if self.needs_calibration:   # population baseline, used until / unless a personal one exists
            self.population = (np.array(meta["pop_median"]), np.array(meta["pop_std"]))

    def predict_frame(self, feats, crop, baseline=None):
        if self.needs_calibration:
            feats = {**feats, **z_features(feats, baseline if baseline is not None else self.population)}
        return float(self.model.predict_proba(np.array([[feats.get(c, 0.0) for c in self.features]], np.float32))[0, 1])


class CnnFaceModel:
    per_frame = True

    def __init__(self, d, meta):
        import torch
        self.torch, self.meta, self.arch, self.img = torch, meta, meta["arch"], meta["img"]
        self.model = build_cnn(self.arch, pretrained=False, img=self.img)
        self.model.load_state_dict(torch.load(d / "model.pt", map_location="cpu"))
        self.model.eval()
        self.needs_calibration = False

    def predict_frame(self, feats, crop, baseline=None):
        with self.torch.no_grad():
            return float(self.torch.softmax(self.model(cnn_preprocess([crop], self.img)), 1)[0, 1])

    def gradcam(self, crop):
        return gradcam(self.model, self.arch, crop, self.img)


class FusionModel:
    per_frame = True

    def __init__(self, d, meta):
        self.meta, self.w = meta, meta["weight"]
        self.a, self.b = load_model(d / "component_a"), load_model(d / "component_b")
        self.needs_calibration = self.a.needs_calibration

    def predict_frame(self, feats, crop, baseline=None):
        return self.w * self.a.predict_frame(feats, crop, baseline) + (1 - self.w) * self.b.predict_frame(feats, crop)

    def gradcam(self, crop):
        return self.b.gradcam(crop) if hasattr(self.b, "gradcam") else None


class WindowModel:
    per_frame = False
    needs_calibration = False

    def __init__(self, d, meta):
        self.meta, self.kind, self.features = meta, meta["kind"], meta["features"]
        self.fps = meta.get("fps", TARGET_FPS)
        self.length = int(round(meta.get("window_s", 10.0) * self.fps))
        if self.kind == "sequence":
            import torch
            self.torch = torch
            self.model = build_sequence(meta["arch"], len(self.features))
            self.model.load_state_dict(torch.load(d / "model.pt", map_location="cpu"))
            self.model.eval()
            n = np.load(d / "norm.npz")
            self.mu, self.sd = n["mean"], n["std"]
        else:
            import joblib
            self.model = joblib.load(d / "model.joblib")

    def predict_window(self, frames):
        if self.kind == "sequence":
            arr = np.array([[f.get(c, 0.0) for c in self.features] for f in frames], np.float32)
            with self.torch.no_grad():
                x = self.torch.from_numpy(((arr - self.mu) / self.sd)[None].astype(np.float32))
                return float(self.torch.softmax(self.model(x), 1)[0, 1])
        agg = window_features({c: np.array([f.get(c, 0.0) for f in frames]) for c in WINDOW_BASE}, self.fps)
        return float(self.model.predict_proba(np.array([[agg[c] for c in self.features]], np.float32))[0, 1])


def load_model(d):
    d = Path(d)
    meta = json.loads((d / "meta.json").read_text())
    return {"classic_frame": ClassicFrameModel, "cnn_face": CnnFaceModel, "fusion": FusionModel,
            "sequence": WindowModel, "classic_window": WindowModel}[meta["kind"]](d, meta)


def gradcam(model, arch, crop, img):
    """Grad-CAM overlay for the 'drowsy' class (None for transformers)."""
    layer = gradcam_layer(model, arch)
    if layer is None:
        return None
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda m, i, o: acts.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("g", go[0]))
    try:
        model.eval()
        x = cnn_preprocess([crop], img).requires_grad_(True)
        out = model(x)
        model.zero_grad()
        out[0, 1].backward()
        cam = (grads["g"][0].mean(dim=(1, 2), keepdim=True) * acts["a"][0]).sum(0).clamp(min=0).detach().numpy()
    finally:
        h1.remove()
        h2.remove()
    cam = cv2.resize(cam / (cam.max() + 1e-8), (crop.shape[1], crop.shape[0]))
    return cv2.addWeighted(crop, 0.55, cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET), 0.45, 0)


# ============================================================================ inference engine
def heuristic_prob(f):
    """Fallback before any model is trained: eye closure + yawning from blendshapes."""
    return float(np.clip(0.75 * (f["bs_eyeBlinkLeft"] + f["bs_eyeBlinkRight"]) / 2 + 0.25 * min(f["bs_jawOpen"] / YAWN_JAW, 1), 0, 1))


class Session:
    def __init__(self, engine, fps=TARGET_FPS):
        self.state, self.smoother = SessionState(fps=fps), Smoother()
        self.window = deque(maxlen=max(engine.window_len, 1))
        self.last_window_prob = None
        self.calib_rows, self.baseline, self.t_start = [], None, None


class DrowsinessEngine:
    def __init__(self, best_dir=BEST_DIR):
        self.face = FaceAnalyzer()
        self.model, self.info = None, {"id": "rules", "name": "Rule-based fallback (no trained model yet)", "kind": "rules"}
        if (Path(best_dir) / "config.json").exists():
            self.model = load_model(best_dir)
            self.info = json.loads((Path(best_dir) / "config.json").read_text())
        self.window_len = getattr(self.model, "length", 0) if self.model is not None and not self.model.per_frame else 0

    def _frame(self, bgr):
        img, q = quality_check(bgr)
        face = self.face.detect(img)
        if face is None:
            return img, q, None, None, None
        return img, q, face, frame_features(face), crop_face(img, face, 160)

    def _prob(self, feats, crop, baseline=None):
        if self.model is None or not self.model.per_frame:
            return heuristic_prob(feats)
        return self.model.predict_frame(feats, crop, baseline)

    @property
    def needs_calibration(self):
        return bool(getattr(self.model, "needs_calibration", False))

    def population_std(self):
        m = self.model.a if hasattr(self.model, "a") else self.model
        return m.population[1]

    @staticmethod
    def snapshot(f):
        eyes = (f["bs_eyeBlinkLeft"] + f["bs_eyeBlinkRight"]) / 2
        return {"ear": round(f["ear_mean"], 4), "mar": round(f["mar"], 4), "eye_closure": round(eyes, 3),
                "jaw_open": round(f["bs_jawOpen"], 3), "pitch": round(f["pitch"], 1), "yaw": round(f["yaw"], 1),
                "roll": round(f["roll"], 1), "eyes_closed": bool(eyes > 0.55 or f["ear_mean"] < EYE_CLOSED_ABS),
                "yawning": bool(f["bs_jawOpen"] > YAWN_JAW)}

    def analyze_image(self, bgr, explain=True, reference_bgr=None):
        """Snapshot check. `reference_bgr` = optional photo of the same person while alert (personal calibration)."""
        t0 = time.perf_counter()
        img, q, face, feats, crop = self._frame(bgr)
        if face is None:
            return {"face": False, "quality": q, "model": self.info, "message": "No face detected"}
        baseline = None
        if reference_bgr is not None and self.needs_calibration:
            ref = self._frame(reference_bgr)[3]
            if ref is not None:
                baseline = (np.array([ref[c] for c in FRAME_FEATURES]), self.population_std())
        snap = self.snapshot(feats)
        p = self._prob(feats, crop, baseline)
        score = 0.75 * p + 0.25 * (0.7 * snap["eyes_closed"] + 0.3 * snap["yawning"])
        out = {"face": True, "score": float(score), "model_prob": float(p), "state": state_of(score), "kss": kss_of(score),
               "kss_label": KSS_LABELS[kss_of(score)], "signals": snap, "quality": q, "reasons": reasons({}, p, snap),
               "overlay": overlay_points(face), "face_crop": crop, "model": self.info, "gradcam": None,
               "calibrated": baseline is not None}
        if explain and hasattr(self.model, "gradcam"):
            out["gradcam"] = self.model.gradcam(crop)
        out["latency_ms"] = (time.perf_counter() - t0) * 1000
        return out

    def new_session(self, fps=TARGET_FPS):
        return Session(self, fps)

    def analyze_stream_frame(self, session, bgr, t):
        t0 = time.perf_counter()
        img, q, face, feats, crop = self._frame(bgr)
        prob = None
        if session.t_start is None:
            session.t_start = t
        if face is not None and self.needs_calibration and session.baseline is None:
            if q["usable"]:
                session.calib_rows.append(feats)        # driver assumed alert during the first seconds
            if t - session.t_start >= session.state.calibration_s and len(session.calib_rows) >= 10:
                session.baseline = calibration_stats(session.calib_rows)
        if face is not None:
            if self.model is not None and not self.model.per_frame:
                session.window.append(feats)
                if len(session.window) == session.window.maxlen:
                    session.last_window_prob = self.model.predict_window(list(session.window))
                prob = session.last_window_prob if session.last_window_prob is not None else heuristic_prob(feats)
            else:
                prob = self._prob(feats, crop, session.baseline)
        s = session.state.update(t, feats, prob, usable=q["usable"] and face is not None)
        s["model_calibrated"] = session.baseline is not None or not self.needs_calibration
        score = session.smoother(combine(s["prob_window"], s)) if face is not None else (session.smoother.value or 0.0)
        out = {"t": t, "face": face is not None, "score": float(score), "state": state_of(score), "kss": kss_of(score),
               "kss_label": KSS_LABELS[kss_of(score)], "model_prob": prob, "temporal": s, "quality": q,
               "alarm": bool(score >= DROWSY_MIN or s["current_closure_s"] >= 1.5),
               "reasons": reasons(s, s["prob_window"]), "latency_ms": (time.perf_counter() - t0) * 1000}
        if face is not None:
            out["signals"], out["overlay"] = self.snapshot(feats), overlay_points(face)
        return out

    def analyze_video(self, path, sample_fps=TARGET_FPS, max_seconds=600, progress=None):
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise ValueError("could not open the video")
        src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        step = max(1, int(round(src_fps / sample_fps)))
        session = self.new_session(src_fps / step)
        session.state.calibration_s = min(CALIBRATION_SECONDS, max(3.0, total / src_fps * 0.15))
        timeline, idx, t0 = [], 0, time.perf_counter()
        while cap.grab():
            t = idx / src_fps
            if t > max_seconds:
                break
            if idx % step == 0:
                ok, frame = cap.retrieve()
                if ok:
                    if frame.shape[1] > 960:
                        frame = cv2.resize(frame, None, fx=960 / frame.shape[1], fy=960 / frame.shape[1], interpolation=cv2.INTER_AREA)
                    r = self.analyze_stream_frame(session, frame, t)
                    sig = r.get("signals") or {}
                    timeline.append({"t": round(t, 2), "score": r["score"], "state": r["state"], "kss": r["kss"],
                                     "model_prob": r["model_prob"], "perclos": r["temporal"]["perclos"],
                                     "ear": sig.get("ear"), "mar": sig.get("mar"), "face": r["face"], "alarm": r["alarm"]})
                    if progress and total:
                        progress(min(idx / total, 1.0))
            idx += 1
        cap.release()
        if not timeline:
            raise ValueError("no frames could be read from the video")
        scores = np.array([p["score"] for p in timeline])
        episodes, cur = [], None
        for p in timeline:
            if p["state"] == "drowsy" and cur is None:
                cur = {"start": p["t"]}
            elif p["state"] != "drowsy" and cur is not None:
                cur["end"] = p["t"]
                episodes.append(cur)
                cur = None
        if cur is not None:
            cur["end"] = timeline[-1]["t"]
            episodes.append(cur)
        overall = float(np.percentile(scores, 90))
        st = session.state
        return {"duration_s": timeline[-1]["t"], "frames_analyzed": len(timeline),
                "face_ratio": float(np.mean([p["face"] for p in timeline])), "processing_s": time.perf_counter() - t0,
                "overall_score": overall, "overall_state": state_of(overall), "kss": kss_of(overall),
                "kss_label": KSS_LABELS[kss_of(overall)], "peak_score": float(scores.max()),
                "drowsy_share": float(np.mean([p["state"] == "drowsy" for p in timeline])), "episodes": episodes,
                "events": {"blinks": st.blinks_total, "long_closures": st.long_closures_total, "yawns": st.yawns_total,
                           "nods": st.nods_total}, "timeline": timeline, "model": self.info}


# ============================================================================ feature-extraction workers
_worker_face = None


def init_worker():
    global _worker_face
    cv2.setNumThreads(1)
    _worker_face = FaceAnalyzer()


def extract_one(job):
    """(image_path, crop_path) -> feature row. Runs in a worker process."""
    path, crop_path = job
    bgr = cv2.imread(path)
    if bgr is None:
        return {"face": False}
    img, q = quality_check(bgr)
    face = _worker_face.detect(img)
    row = {"blur": q["blur"], "brightness": q["brightness"], "usable": q["usable"], "face": face is not None}
    if face is not None:
        row.update(frame_features(face))
        Path(crop_path).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(crop_path, crop_face(img, face, 160), [cv2.IMWRITE_JPEG_QUALITY, 92])
    return row


def extract_video(job):
    """(video_row_dict, fps) -> DataFrame of per-frame features for one video (UTA-RLDD)."""
    import pandas as pd
    row, fps = job
    cap = cv2.VideoCapture(row["path"])
    src = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step, out, idx = max(1, int(round(src / fps))), [], 0
    while cap.grab():
        if idx % step == 0:
            ok, frame = cap.retrieve()
            if ok:
                if frame.shape[1] > 960:
                    frame = cv2.resize(frame, None, fx=960 / frame.shape[1], fy=960 / frame.shape[1])
                img, q = quality_check(frame)
                face = _worker_face.detect(img)
                r = {"t": idx / src, "face": face is not None, "usable": q["usable"]}
                if face is not None:
                    r.update(frame_features(face))
                out.append(r)
        idx += 1
    cap.release()
    df = pd.DataFrame(out)
    for k in ("subject_id", "video_id", "label", "label3"):
        df[k] = row[k]
    return df

In [ ]:
import importlib, sys
sys.path.insert(0, str(PROJECT))
import drowsy_core as core
importlib.reload(core)
print("core pipeline loaded:", len(core.FRAME_FEATURES), "per-frame features")

### 2.9 Pipeline check on one image
Run one drowsy and one alert DDD image through the pipeline: landmarks, eye and mouth contours, and the main features.

In [ ]:
analyzer = core.FaceAnalyzer()
DDD_IMG = DDD_ROOT / "Driver Drowsiness Dataset (DDD)"
examples = [("Drowsy", DDD_IMG / "Drowsy" / "A0100.png"), ("Non drowsy", DDD_IMG / "Non Drowsy" / "a0100.png")]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, (name, p) in zip(axes, examples):
    img, q = core.quality_check(cv2.imread(str(p)))
    face = analyzer.detect(img)
    f = core.frame_features(face)
    vis = core.draw_overlay(img, core.overlay_points(face))
    for x, y, _ in face.points[::4]:
        cv2.circle(vis, (int(x), int(y)), 1, (0, 255, 0), -1)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"{name}\nEAR {f['ear_mean']:.3f} · MAR {f['mar']:.3f} · blink {f['bs_eyeBlinkLeft']:.2f} · pitch {f['pitch']:.0f}°", fontsize=9)
plt.tight_layout(); plt.show()

## 3. Feature extraction (all 41,793 images)
Every image goes through the same pipeline as the app: quality gate → landmarks → 61 features + a 160 px face crop for the CNNs. It runs in parallel worker processes and is cached in `data/processed/ddd/frames.csv`; a re-run loads the cache instead of extracting again.

In [ ]:
import re, multiprocessing as mp
from tqdm.auto import tqdm

def load_ddd(root):
    rows = []
    for p in sorted(Path(root).rglob("*.png")):
        parent = p.parent.name.lower()
        label = 0 if parent.startswith("non") else (1 if "drows" in parent else None)
        if label is None:
            continue
        rows.append({"path": str(p), "subject_id": re.match(r"([A-Za-z]+)", p.stem).group(1).upper(),
                     "label": label, "sample_id": f"{label}_{p.stem}"})
    return pd.DataFrame(rows)

FRAMES_CSV = core.PROCESSED_DIR / "ddd" / "frames.csv"
ddd = load_ddd(DDD_ROOT)
if FRAMES_CSV.exists() and len(pd.read_csv(FRAMES_CSV, usecols=["sample_id"])) == len(ddd):
    frames = pd.read_csv(FRAMES_CSV)
    print(f"loaded cached features: {FRAMES_CSV}")
else:
    core.ensure_landmarker_model()
    crops = core.PROCESSED_DIR / "ddd" / "crops"
    ddd["crop_path"] = [str(crops / str(l) / f"{s}.jpg") for s, l in zip(ddd["sample_id"], ddd["label"])]
    t0 = time.time()
    with mp.Pool(EXTRACT_WORKERS, initializer=core.init_worker) as pool:
        rows = list(tqdm(pool.imap(core.extract_one, zip(ddd["path"], ddd["crop_path"]), chunksize=16), total=len(ddd)))
    frames = pd.concat([ddd, pd.DataFrame(rows)], axis=1)
    frames.loc[~frames["face"].astype(bool), "crop_path"] = ""
    frames.to_csv(FRAMES_CSV, index=False)
    print(f"extracted in {(time.time() - t0) / 60:.1f} min")
print(f"{len(frames)} images · {frames['subject_id'].nunique()} participants · face found in {frames['face'].mean():.2%} · "
      f"usable quality {frames['usable'].mean():.2%}")

### 3.1 Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1]})
per = frames.groupby(["subject_id", "label"]).size().unstack(fill_value=0).rename(columns={0: "non-drowsy", 1: "drowsy"})
per.plot.bar(stacked=True, ax=axes[0], color=["#16a34a", "#dc2626"], width=0.8)
axes[0].set_title("Images per participant"); axes[0].set_xlabel("participant")
frames["label"].map({0: "non-drowsy", 1: "drowsy"}).value_counts().plot.pie(ax=axes[1], autopct="%1.0f%%", colors=["#dc2626", "#16a34a"])
axes[1].set_ylabel(""); axes[1].set_title("Class balance")
plt.tight_layout(); plt.show()

In [ ]:
show = ["ear_mean", "mar", "bs_eyeBlinkLeft", "bs_eyeSquintLeft", "bs_jawOpen", "pitch", "bs_eyeLookDownLeft", "bs_browInnerUp"]
fd = frames[frames["face"].astype(bool)].assign(state=lambda d: d["label"].map({0: "non-drowsy", 1: "drowsy"}))
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, c in zip(axes.ravel(), show):
    sns.violinplot(data=fd.sample(8000, random_state=0), x="state", y=c, ax=ax, palette=["#16a34a", "#dc2626"],
                   order=["non-drowsy", "drowsy"], cut=0, inner="quartile")
    ax.set_xlabel(""); ax.set_title(c)
plt.suptitle("Feature distributions by class: drowsy faces have lower EAR and higher blink / squint / jaw-open scores", y=1.01)
plt.tight_layout(); plt.show()

## 4. Subject-disjoint split and calibration frames
**Personal calibration frames.** The app learns each driver's baseline from the first seconds of a drive, when the driver is assumed to be alert. Here that is mirrored as follows: for every participant, the **first 20 % of their alert frames** (in recording order) become calibration frames. Their per-feature median and standard deviation define that person's baseline, and every frame is turned into z-scores against it (`z_*` features). Calibration frames are **removed from training and from testing for every model**, so all models are scored on exactly the same frames and none is scored on frames used to calibrate it.

Participants, not images, are divided into **train (≈70 %) / validation (≈15 %) / test (≈15 %)**. **Every model is evaluated on the same test participants**, none of whom appear in training. The split is saved to `data/splits/ddd.json` so it stays the same across runs and machines. The classic models also get 5-fold `GroupKFold` cross-validation over the train + validation participants, to show how much results vary from person to person.

In [ ]:
SPLIT_FILE = core.SPLITS_DIR / "ddd.json"
if SPLIT_FILE.exists():
    split = json.loads(SPLIT_FILE.read_text())
else:
    subs = sorted(frames["subject_id"].unique())
    np.random.default_rng(core.SEED).shuffle(subs)
    n_test = n_val = max(1, round(len(subs) * 0.15))
    split = {"test": sorted(subs[:n_test]), "val": sorted(subs[n_test:n_test + n_val]), "train": sorted(subs[n_test + n_val:])}
    SPLIT_FILE.write_text(json.dumps(split, indent=2))
lookup = {s: k for k in ("train", "val", "test") for s in split[k]}
data = frames[frames["face"].astype(bool)].copy()
data["split"] = data["subject_id"].map(lookup)

# personal-calibration frames: the first 20 % of each participant's alert frames, in recording order
F, Z = core.FRAME_FEATURES, core.Z_FEATURES
data["order"] = data["path"].str.extract(r"(\d+)\.png$")[0].astype(int)
data = data.sort_values(["subject_id", "label", "order"])
data["calib"] = False
for s, g in data[data["label"] == 0].groupby("subject_id"):
    data.loc[g.index[: max(30, int(0.2 * len(g)))], "calib"] = True
base = data[data["calib"]].groupby("subject_id")[F]
CAL_MED, CAL_SD = base.median(), base.std().fillna(0) + 1e-3
has_cal = data["subject_id"].isin(CAL_MED.index)
data[Z] = np.nan
data.loc[has_cal, Z] = np.clip((data.loc[has_cal, F].to_numpy() - CAL_MED.loc[data.loc[has_cal, "subject_id"]].to_numpy())
                               / CAL_SD.loc[data.loc[has_cal, "subject_id"]].to_numpy(), -50, 50)
data.loc[data["calib"], "split"] = "calib"          # never used for training or scoring
data_cal = data[has_cal]                             # participants with alert frames (needed for calibration)
train_subs = [s for s in split["train"] if s in CAL_MED.index]
CAL_META = {"calibrated": True, "pop_median": CAL_MED.loc[train_subs].median().tolist(), "pop_std": CAL_SD.loc[train_subs].median().tolist()}

summary = data.groupby("split").agg(participants=("subject_id", "nunique"), images=("label", "size"), drowsy_share=("label", "mean"))
display(summary.loc[["train", "val", "test", "calib"]].style.format({"drowsy_share": "{:.1%}"}))
print("test participants:", split["test"], "| participants without alert frames (no calibration possible):",
      sorted(set(data["subject_id"]) - set(CAL_MED.index)))

## 5. The model zoo
Every model writes the same result record (accuracy, precision, recall, F1, ROC-AUC, confusion matrix, ROC curve, CPU latency, size) to `reports/results/<id>.json`. That record drives the comparison and the automatic selection. A model that is already trained is skipped unless `FORCE_RETRAIN = True`.

| Group | Input | Models |
|---|---|---|
| **Paper** | EAR + MAR of one frame | Decision tree, neural network, linear SVM, LDA (reference setup) |
| **A** | 61 frame features | LogReg, KNN, Naive Bayes, LDA, Decision tree, Linear SVM, RBF SVM, Random forest, Extra trees, HistGB, XGBoost, LightGBM, CatBoost, MLP |
| **A+** | 61 frame features + 61 personal-calibration z-scores | the same 14 classifiers (ablation: what calibration adds) |
| **B** | 10 s window features (video) | the same 14 classifiers on PERCLOS / blink / yawn / pose statistics |
| **C** | face crop | Custom CNN, MobileNetV3-S/L, EfficientNet-B0, ResNet-18/50, VGG-16, DeiT-Tiny |
| **D** | 10 s feature sequences (video) | 1D-CNN, TCN, LSTM, BiLSTM, GRU, Transformer |
| **E** | combinations | Stacking (RF + XGB + LGBM → LogReg), late fusion (best A + best C) |

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold
import joblib

GROUP_TITLES = {"paper": "Reference paper baseline", "A": "Classic ML · frame features",
                "A+": "Classic ML · frame + personal calibration", "B": "Classic ML · window features",
                "C": "CNN / ViT · face crops", "D": "Sequence models", "E": "Ensembles & fusion"}
CLASSIC = [("logreg", "Logistic Regression"), ("knn", "K-Nearest Neighbours"), ("gnb", "Gaussian Naive Bayes"),
           ("lda", "Linear Discriminant Analysis"), ("dtree", "Decision Tree"), ("linsvm", "Linear SVM"), ("rbfsvm", "RBF SVM"),
           ("rf", "Random Forest"), ("extratrees", "Extra Trees"), ("hgb", "Gradient Boosting (hist)"), ("xgb", "XGBoost"),
           ("lgbm", "LightGBM"), ("catboost", "CatBoost"), ("mlp", "MLP neural network")]
CNNS = [("custom_cnn", "Custom CNN (from scratch)", "custom", 112, False), ("mobilenetv3_small", "MobileNetV3-Small", "mobilenetv3_small_100", 128, False),
        ("mobilenetv3_large", "MobileNetV3-Large", "mobilenetv3_large_100", 128, False), ("efficientnet_b0", "EfficientNet-B0", "efficientnet_b0", 128, False),
        ("resnet18", "ResNet-18", "resnet18", 128, False), ("resnet50", "ResNet-50", "resnet50", 160, True),
        ("vgg16", "VGG-16 (BN)", "vgg16_bn", 128, True), ("deit_tiny", "DeiT-Tiny (Vision Transformer)", "deit_tiny_patch16_224", 224, True)]
SEQS = [("conv1d", "1D-CNN"), ("tcn", "Temporal Convolutional Network"), ("lstm", "LSTM"), ("bilstm", "BiLSTM"), ("gru", "GRU"), ("transformer", "Transformer encoder")]

REGISTRY = ([{"id": f"paper_{k}", "name": f"Paper: {n} (EAR+MAR)", "group": "paper"} for k, n in
             [("dtree", "Decision Tree"), ("mlp", "Neural Network"), ("linsvm", "Linear SVM"), ("lda", "LDA")]]
            + [{"id": f"A_{k}", "name": n, "group": "A"} for k, n in CLASSIC]
            + [{"id": f"Acal_{k}", "name": f"{n} + calibration", "group": "A+"} for k, n in CLASSIC]
            + [{"id": f"B_{k}", "name": n, "group": "B"} for k, n in CLASSIC]
            + [{"id": f"C_{k}", "name": n, "group": "C", "heavy": h} for k, n, _, _, h in CNNS]
            + [{"id": f"D_{k}", "name": n, "group": "D"} for k, n in SEQS]
            + [{"id": "E_stacking", "name": "Stacking (RF + XGB + LGBM → LogReg, calibrated features)", "group": "E"},
               {"id": "E_late_fusion", "name": "Late fusion (best landmark model + best CNN)", "group": "E"}])
REG = {r["id"]: r for r in REGISTRY}


def metrics(y, prob, thr=0.5):
    y, prob = np.asarray(y).astype(int), np.asarray(prob, float)
    pred = (prob >= thr).astype(int)
    fpr, tpr, _ = roc_curve(y, prob)
    k = np.linspace(0, len(fpr) - 1, min(80, len(fpr))).astype(int)
    return {"accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, prob), "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
            "roc": {"fpr": fpr[k].round(4).tolist(), "tpr": tpr[k].round(4).tolist()}, "n_test": int(len(y))}


def latency_ms(fn, x, repeats=50):
    for _ in range(5):
        fn(x)
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(x); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))


def done(mid):
    return not FORCE_RETRAIN and (core.RESULTS_DIR / f"{mid}.json").exists()


def save_preds(mid, part, ids, y, prob):
    pd.DataFrame({"sample_id": list(ids), "y_true": np.asarray(y).astype(int), "prob": np.asarray(prob, float)}) \
      .to_csv(core.PREDICTIONS_DIR / f"{mid}__{part}.csv", index=False)


def save_result(mid, kind, m, lat, artifact, **extra):
    size = sum(p.stat().st_size for p in Path(artifact).rglob("*") if p.is_file()) / 1e6
    r = {"id": mid, "name": REG[mid]["name"], "group": REG[mid]["group"], "kind": kind, "metrics": m,
         "latency_ms": lat, "size_mb": size, "artifact": str(Path(artifact).relative_to(PROJECT)), **extra}
    (core.RESULTS_DIR / f"{mid}.json").write_text(json.dumps(r, indent=2, default=float))
    cv = f" | CV F1 {extra['cv']['f1_mean']:.3f} ± {extra['cv']['f1_std']:.3f}" if extra.get("cv") else ""
    print(f"{mid:<24} acc {m['accuracy']:.3f}  F1 {m['f1']:.3f}  AUC {m['roc_auc']:.3f}  {lat:7.2f} ms{cv}")
    return r

### 5.1 Reference-paper baseline and Group A (classic ML on frame features)
Same split for every model. The paper baseline sees only EAR and MAR; Group A sees all 61 features. Each classic model also gets 5-fold subject-grouped CV. The RBF SVM is trained on a random 8,000-sample subset because its cost grows quadratically with the number of samples.

In [ ]:
def train_classic(mid, df, features, kind="classic_frame", max_train=None, cv=RUN_CV, extra_meta=None):
    if done(mid):
        print(f"{mid:<24} cached"); return
    tr, va, te = (df[df["split"] == s] for s in ("train", "val", "test"))
    if max_train and len(tr) > max_train:
        tr = tr.sample(max_train, random_state=core.SEED)
    t0 = time.time()
    model = core.classic_pipeline(mid).fit(tr[features].to_numpy(np.float32), tr["label"].to_numpy())
    fit_s = time.time() - t0
    p_te = model.predict_proba(te[features].to_numpy(np.float32))[:, 1]
    p_va = model.predict_proba(va[features].to_numpy(np.float32))[:, 1]
    save_preds(mid, "test", te["sample_id"], te["label"], p_te)
    save_preds(mid, "val", va["sample_id"], va["label"], p_va)
    cvres = None
    if cv:
        tv = df[df["split"].isin(["train", "val"])]
        f1s, accs = [], []
        for a, b in GroupKFold(5).split(tv, tv["label"], tv["subject_id"]):
            fa = tv.iloc[a]
            if max_train and len(fa) > max_train:
                fa = fa.sample(max_train, random_state=core.SEED)
            mm = core.classic_pipeline(mid).fit(fa[features].to_numpy(np.float32), fa["label"].to_numpy())
            pr = mm.predict(tv.iloc[b][features].to_numpy(np.float32))
            f1s.append(f1_score(tv.iloc[b]["label"], pr)); accs.append(accuracy_score(tv.iloc[b]["label"], pr))
        cvres = {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)), "accuracy_mean": float(np.mean(accs))}
    out = core.CANDIDATES_DIR / mid
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, out / "model.joblib", compress=3)
    (out / "meta.json").write_text(json.dumps({"id": mid, "kind": kind, "features": features, **(extra_meta or {})}))
    x1 = te[features].to_numpy(np.float32)[:1]
    save_result(mid, kind, metrics(te["label"], p_te), latency_ms(model.predict_proba, x1), out, cv=cvres,
                fit_seconds=fit_s, n_train=len(tr), n_features=len(features))


for r in REGISTRY:
    if r["group"] == "paper":
        train_classic(r["id"], data, core.PAPER_FEATURES)
for r in REGISTRY:
    if r["group"] == "A":
        train_classic(r["id"], data, core.FRAME_FEATURES, max_train=8000 if r["id"] == "A_rbfsvm" else None)

### 5.2 Group A+: personal calibration
The same 14 classifiers, now given each person's **z-scores against their own alert baseline** in addition to the raw features. One driver's normal eye openness can look like another driver's drowsy eyes; calibration removes that between-person difference. Raw Group A vs Group A+ is therefore a direct ablation of the calibration idea. Participants without any alert frames can't be calibrated and are left out of this group (none of them is in the test set).

In [ ]:
for r in REGISTRY:
    if r["group"] == "A+":
        train_classic(r["id"], data_cal, F + Z, max_train=8000 if r["id"] == "Acal_rbfsvm" else None, extra_meta=CAL_META)

### 5.3 Group C: CNN / ViT image models on face crops
Pretrained backbones are fine-tuned with AdamW, a one-cycle learning-rate schedule, class-weighted cross-entropy with label smoothing, and augmentation (random crop, flip, colour jitter, blur, greyscale, rotation). The checkpoint with the best **validation** F1 is kept; the test participants are used only for the final score.

**CPU budget:** this laptop has no GPU, so training uses every `CNN_TRAIN_STRIDE`-th frame (consecutive frames are near-duplicates) for `CNN_EPOCHS` epochs. The heavy models get an even smaller budget, and the `budget` column records this. On a GPU, raise the epochs and set the strides to 1.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
torch.manual_seed(core.SEED)
torch.set_num_threads(os.cpu_count() or 4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def cnn_predict(model, loader):
    model.eval()
    return torch.cat([torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu() for x, _ in loader]).numpy()


def train_cnn(key, name, arch, img, heavy):
    mid = f"C_{key}"
    if done(mid):
        print(f"{mid:<24} cached"); return
    stride, epochs = (HEAVY_TRAIN_STRIDE, HEAVY_EPOCHS) if heavy else (CNN_TRAIN_STRIDE, CNN_EPOCHS)
    df = data[data["crop_path"].fillna("").ne("")]
    tr = df[df["split"] == "train"]
    tr = tr[tr.groupby(["subject_id", "label"]).cumcount() % stride == 0]
    va, te = df[df["split"] == "val"], df[df["split"] == "test"]
    mk = lambda d, train: DataLoader(core.CropDataset(d["crop_path"], d["label"], img, train), batch_size=64,
                                     shuffle=train, num_workers=2, persistent_workers=True)
    tr_dl, va_dl, te_dl = mk(tr, True), mk(va, False), mk(te, False)
    model = core.build_cnn(arch, pretrained=True, img=img).to(DEVICE)
    lr = 1e-3 if arch == "custom" else 5e-4
    pos = tr["label"].mean()
    crit = nn.CrossEntropyLoss(weight=torch.tensor([1.0, (1 - pos) / pos], device=DEVICE, dtype=torch.float32), label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tr_dl))
    out = core.CANDIDATES_DIR / mid
    out.mkdir(parents=True, exist_ok=True)
    best, history, t0 = -1, [], time.time()
    for ep in range(epochs):
        model.train(); run = 0.0
        for x, y in tqdm(tr_dl, desc=f"{mid} epoch {ep + 1}/{epochs}", leave=False):
            opt.zero_grad()
            loss = crit(model(x.to(DEVICE)), y.to(DEVICE))
            loss.backward(); opt.step(); sched.step(); run += loss.item()
        vm = metrics(va["label"], cnn_predict(model, va_dl))
        history.append({"epoch": ep + 1, "loss": run / len(tr_dl), "val_acc": vm["accuracy"], "val_f1": vm["f1"]})
        print(f"  {mid} epoch {ep + 1}: loss {run / len(tr_dl):.3f} · val acc {vm['accuracy']:.3f} · val F1 {vm['f1']:.3f}")
        if vm["f1"] > best:
            best = vm["f1"]; torch.save(model.state_dict(), out / "model.pt")
    model.load_state_dict(torch.load(out / "model.pt", map_location=DEVICE))
    p_va, p_te = cnn_predict(model, va_dl), cnn_predict(model, te_dl)
    save_preds(mid, "val", va["sample_id"], va["label"], p_va)
    save_preds(mid, "test", te["sample_id"], te["label"], p_te)
    (out / "meta.json").write_text(json.dumps({"id": mid, "kind": "cnn_face", "arch": arch, "img": img}))
    cpu = model.to("cpu").eval()
    with torch.no_grad():
        lat = latency_ms(cpu, torch.randn(1, 3, img, img), repeats=20)
    save_result(mid, "cnn_face", metrics(te["label"], p_te), lat, out, history=history, fit_seconds=time.time() - t0,
                n_train=len(tr), epochs=epochs, budget=f"{epochs} ep · every {stride}th frame",
                params_m=sum(p.numel() for p in cpu.parameters()) / 1e6)


for key, name, arch, img, heavy in CNNS:
    if heavy and not TRAIN_HEAVY_CNN:
        continue
    try:
        train_cnn(key, name, arch, img, heavy)
    except Exception as e:
        print(f"C_{key}: failed — {e}")

### 5.4 Group E: stacking and late fusion
* **Stacking**: Random Forest, XGBoost and LightGBM feed out-of-fold probabilities to a logistic-regression meta-learner.
* **Late fusion**: `p = w·p(best landmark model, raw or calibrated) + (1−w)·p(best CNN)`. The two members are chosen by validation F1, and `w` is tuned on the **validation** participants only. Landmarks capture geometry; the CNN captures appearance, and it still works when landmarks are noisy (blur, glasses).

In [ ]:
train_classic("E_stacking", data_cal, F + Z, cv=False, extra_meta=CAL_META)


def val_f1(mid):
    p = core.PREDICTIONS_DIR / f"{mid}__val.csv"
    if not p.exists():
        return -1
    d = pd.read_csv(p)
    return f1_score(d["y_true"], d["prob"] >= 0.5)


def late_fusion():
    if done("E_late_fusion"):
        print("E_late_fusion            cached"); return
    res = {p.stem: json.loads(p.read_text()) for p in core.RESULTS_DIR.glob("*.json")}
    a = max((m for m in res if m.startswith(("A_", "Acal_"))), key=val_f1)
    c = max((m for m in res if m.startswith("C_")), key=val_f1)
    def merged(part):
        pa = pd.read_csv(core.PREDICTIONS_DIR / f"{a}__{part}.csv").set_index("sample_id")
        pc = pd.read_csv(core.PREDICTIONS_DIR / f"{c}__{part}.csv").set_index("sample_id")
        j = pa.join(pc, lsuffix="_a", rsuffix="_c", how="inner")
        return j.index, j["y_true_a"].to_numpy(), j["prob_a"].to_numpy(), j["prob_c"].to_numpy()
    _, yv, av, cv_ = merged("val")
    ws = np.round(np.arange(0, 1.01, 0.1), 1)
    w = float(ws[np.argmax([f1_score(yv, (x * av + (1 - x) * cv_) >= 0.5) for x in ws])])
    ids, yt, at, ct = merged("test")
    prob = w * at + (1 - w) * ct
    save_preds("E_late_fusion", "test", ids, yt, prob)
    out = core.CANDIDATES_DIR / "E_late_fusion"
    shutil.rmtree(out, ignore_errors=True)
    shutil.copytree(PROJECT / res[a]["artifact"], out / "component_a")
    shutil.copytree(PROJECT / res[c]["artifact"], out / "component_b")
    (out / "meta.json").write_text(json.dumps({"id": "E_late_fusion", "kind": "fusion", "weight": w, "components": [a, c]}))
    REG["E_late_fusion"]["name"] = f"Late fusion ({res[a]['name']} + {res[c]['name']})"
    save_result("E_late_fusion", "fusion", metrics(yt, prob), res[a]["latency_ms"] + res[c]["latency_ms"], out,
                components=[a, c], weight_landmark_model=w)
    print(f"fusion members: {a} + {c}, landmark weight w = {w}")


late_fusion()

### 5.5 Groups B and D: temporal models on video (UTA-RLDD)
DDD contains single images, so window features (B) and sequence models (D) need real video. [UTA-RLDD](https://sites.google.com/view/utarldd/home) has 60 participants, each with an alert, a low-vigilance and a drowsy video. To use it, put the folds in `data/raw/rldd/` and set `USE_RLDD = True`. The code below then extracts per-frame features at 10 fps, cuts 10 s windows (2.5 s stride) and trains all 14 Group B and 6 Group D models on the same kind of participant-disjoint split. Binary task: alert vs drowsy (low-vigilance videos excluded).

In [ ]:
WINDOW_S, STRIDE_S, RLDD_FPS = 10.0, 2.5, 10


def load_rldd(root):
    rows = []
    for p in sorted(Path(root).rglob("*")):
        m = re.match(r"(\d+)", p.stem)
        if p.suffix.lower() in {".mov", ".mp4", ".avi", ".m4v"} and m and int(m.group(1)) in (0, 5, 10):
            fold = next((x for x in p.parts if x.lower().startswith("fold")), "fold")
            lvl = int(m.group(1))
            rows.append({"path": str(p), "subject_id": f"{fold}_{p.parent.name}", "video_id": f"{fold}_{p.parent.name}_{lvl}",
                         "label3": {0: 0, 5: 1, 10: 2}[lvl], "label": int(lvl == 10)})
    return pd.DataFrame(rows)


def build_windows(fr, fps=RLDD_FPS):
    T, S = int(WINDOW_S * fps), int(STRIDE_S * fps)
    metas, aggs, seqs = [], [], []
    for vid, g in fr.sort_values(["video_id", "t"]).groupby("video_id"):
        if int(g["label3"].iloc[0]) == 1:
            continue
        face = g["face"].astype(bool).to_numpy()
        f = g[core.FRAME_FEATURES].astype(float).where(face[:, None]).interpolate(limit_direction="both").fillna(0)
        arr = f.to_numpy(np.float32)
        for s in range(0, len(g) - T + 1, S):
            if face[s:s + T].mean() < 0.7:
                continue
            aggs.append(core.window_features({c: f[c].to_numpy()[s:s + T] for c in core.WINDOW_BASE}, fps))
            seqs.append(arr[s:s + T])
            metas.append({"sample_id": f"{vid}@{s}", "subject_id": g["subject_id"].iloc[0], "label": int(g["label"].iloc[0])})
    return pd.DataFrame(metas), pd.DataFrame(aggs), np.stack(seqs)


def train_sequence(key, meta, seqs):
    mid = f"D_{key}"
    if done(mid):
        print(f"{mid:<24} cached"); return
    m = {s: (meta["split"] == s).to_numpy() for s in ("train", "val", "test")}
    mu, sd = seqs[m["train"]].reshape(-1, seqs.shape[-1]).mean(0), seqs[m["train"]].reshape(-1, seqs.shape[-1]).std(0) + 1e-6
    X, y = torch.from_numpy(((seqs - mu) / sd).astype(np.float32)), torch.from_numpy(meta["label"].to_numpy())
    dl = {s: DataLoader(torch.utils.data.TensorDataset(X[k], y[k]), batch_size=64, shuffle=s == "train") for s, k in m.items()}
    model = core.build_sequence(key, X.shape[-1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    pred = lambda d: (model.eval(), torch.cat([torch.softmax(model(xb), 1)[:, 1] for xb, _ in d]).detach().numpy())[1]
    out, best, history = core.CANDIDATES_DIR / mid, -1, []
    out.mkdir(parents=True, exist_ok=True)
    for ep in range(40):
        model.train()
        for xb, yb in dl["train"]:
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        f = metrics(y[m["val"]].numpy(), pred(dl["val"]))["f1"]
        history.append({"epoch": ep + 1, "val_f1": f})
        if f > best:
            best = f; torch.save(model.state_dict(), out / "model.pt")
    model.load_state_dict(torch.load(out / "model.pt"))
    p_te = pred(dl["test"])
    save_preds(mid, "test", meta.loc[m["test"], "sample_id"], y[m["test"]].numpy(), p_te)
    np.savez(out / "norm.npz", mean=mu, std=sd)
    (out / "meta.json").write_text(json.dumps({"id": mid, "kind": "sequence", "arch": key, "features": core.FRAME_FEATURES,
                                               "fps": RLDD_FPS, "window_s": WINDOW_S}))
    with torch.no_grad():
        lat = latency_ms(model, X[:1], repeats=20)
    save_result(mid, "sequence", metrics(y[m["test"]].numpy(), p_te), lat, out, history=history)


RLDD_ROOT = PROJECT / "data" / "raw" / "rldd"
if USE_RLDD and RLDD_ROOT.exists():
    vids = load_rldd(RLDD_ROOT)
    vcsv = core.PROCESSED_DIR / "rldd" / "frames.csv"
    if not vcsv.exists():
        vcsv.parent.mkdir(parents=True, exist_ok=True)
        with mp.Pool(EXTRACT_WORKERS, initializer=core.init_worker) as pool:
            parts = list(tqdm(pool.imap(core.extract_video, [(r.to_dict(), RLDD_FPS) for _, r in vids.iterrows()]), total=len(vids)))
        pd.concat(parts).to_csv(vcsv, index=False)
    meta, agg, seqs = build_windows(pd.read_csv(vcsv))
    subs = sorted(meta["subject_id"].unique()); np.random.default_rng(core.SEED).shuffle(subs)
    n = max(1, round(len(subs) * 0.15))
    rl = {s: ("test" if i < n else "val" if i < 2 * n else "train") for i, s in enumerate(subs)}
    meta["split"] = meta["subject_id"].map(rl)
    wdf = pd.concat([meta, agg], axis=1)
    for k, _ in CLASSIC:
        train_classic(f"B_{k}", wdf, list(agg.columns), kind="classic_window", max_train=8000 if k == "rbfsvm" else None,
                      extra_meta={"fps": RLDD_FPS, "window_s": WINDOW_S})
    for k, _ in SEQS:
        train_sequence(k, meta, seqs)
else:
    print("UTA-RLDD not used: Groups B and D are listed as 'needs video data' in the comparison.")

## 6. Model comparison
All numbers below come from the **same unseen test participants**.

In [ ]:
GROUP_COLORS = {"paper": "#9ca3af", "A": "#2563eb", "A+": "#ea580c", "B": "#0891b2", "C": "#7c3aed", "D": "#db2777", "E": "#059669"}
results = {p.stem: json.loads(p.read_text()) for p in core.RESULTS_DIR.glob("*.json")}
rows = []
for r in REGISTRY:
    res = results.get(r["id"])
    if res is None:
        why = "needs video data (UTA-RLDD)" if r["group"] in "BD" else "not trained"
        rows.append({"id": r["id"], "model": r["name"], "group": r["group"], "status": why})
        continue
    m = res["metrics"]
    rows.append({"id": r["id"], "model": res["name"], "group": r["group"], "status": "trained", "accuracy": m["accuracy"],
                 "precision": m["precision"], "recall": m["recall"], "f1": m["f1"], "roc_auc": m["roc_auc"],
                 "cv_f1": (res.get("cv") or {}).get("f1_mean"), "cv_f1_std": (res.get("cv") or {}).get("f1_std"),
                 "latency_ms": res["latency_ms"], "size_mb": res["size_mb"], "budget": res.get("budget", "full"), "kind": res["kind"]})
board = pd.DataFrame(rows)
trained = board[board["status"] == "trained"].sort_values("f1", ascending=False).reset_index(drop=True)
fmt = {c: "{:.1%}" for c in ["accuracy", "precision", "recall", "f1", "cv_f1"]} | {"roc_auc": "{:.3f}", "latency_ms": "{:.2f}", "size_mb": "{:.1f}", "cv_f1_std": "{:.1%}"}
display(trained.drop(columns=["status", "kind"]).style.format(fmt, na_rep="–")
        .background_gradient(subset=["accuracy", "f1", "roc_auc"], cmap="Greens")
        .set_caption(f"{len(trained)} trained models, sorted by F1 on unseen test participants"))
print("Not trained:", ", ".join(board.loc[board["status"] != "trained", "id"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(5, 0.33 * len(trained))))
t = trained.sort_values("accuracy")
ax.barh(t["model"], t["accuracy"], color=[GROUP_COLORS[g] for g in t["group"]], alpha=0.35, label="Accuracy")
ax.barh(t["model"], t["f1"], color=[GROUP_COLORS[g] for g in t["group"]], height=0.45, label="F1")
for i, (a, f) in enumerate(zip(t["accuracy"], t["f1"])):
    ax.text(max(a, f) + 0.005, i, f"{a:.1%} / {f:.1%}", va="center", fontsize=8)
ax.set_xlim(0, 1.08); ax.set_xlabel("score on unseen test participants (wide bar = accuracy, narrow bar = F1)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for g, c in GROUP_COLORS.items() if g in set(t["group"])]
ax.legend(handles, [GROUP_TITLES[g] for g in GROUP_COLORS if g in set(t["group"])], loc="lower right", fontsize=8)
ax.set_title("Accuracy and F1 of every model")
plt.tight_layout(); plt.savefig(core.FIGURES_DIR / "leaderboard.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for g, sub in trained.groupby("group"):
    axes[0].scatter(sub["latency_ms"], sub["f1"], s=70, color=GROUP_COLORS[g], label=GROUP_TITLES[g], edgecolor="white")
    for _, r in sub.iterrows():
        axes[0].annotate(r["id"].split("_", 1)[1], (r["latency_ms"], r["f1"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[0].axvline(core.MAX_MODEL_LATENCY_MS, ls="--", color="#ef4444"); axes[0].set_xscale("log")
axes[0].text(core.MAX_MODEL_LATENCY_MS * 1.05, axes[0].get_ylim()[0] + 0.01, "real-time limit", color="#ef4444")
axes[0].set_xlabel("model latency per frame, CPU (ms, log scale)"); axes[0].set_ylabel("F1 (unseen participants)")
axes[0].set_title("Accuracy vs speed"); axes[0].legend(fontsize=8)
cvd = trained.dropna(subset=["cv_f1"]).sort_values("cv_f1")
axes[1].barh(cvd["model"], cvd["cv_f1"], xerr=cvd["cv_f1_std"], color=[GROUP_COLORS[g] for g in cvd["group"]], capsize=3)
axes[1].set_xlim(0, 1); axes[1].set_title("5-fold subject-grouped CV F1 (mean ± std)")
plt.tight_layout(); plt.savefig(core.FIGURES_DIR / "f1_vs_latency.png", dpi=150); plt.show()

In [ ]:
top = trained.head(6)
fig, axes = plt.subplots(1, len(top), figsize=(3.2 * len(top), 3.2))
for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
    cm = np.array(results[r["id"]]["metrics"]["confusion_matrix"])
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=cm, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["alert", "drowsy"], yticklabels=["alert", "drowsy"])
    ax.set_title(r["model"][:28], fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.suptitle("Confusion matrices of the 6 best models (colour = row-normalised)", y=1.05); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for g, sub in trained.groupby("group"):
    r = results[sub.sort_values("f1").iloc[-1]["id"]]
    ax.plot(r["metrics"]["roc"]["fpr"], r["metrics"]["roc"]["tpr"], color=GROUP_COLORS[g], lw=2,
            label=f"{r['name'][:30]} (AUC {r['metrics']['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="#cbd5e1"); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("ROC: best model of each group"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 6.1 What the models learned
Feature importance for the best tree-based landmark model, Grad-CAM heatmaps for the best CNN, and validation curves for the CNNs.

In [ ]:
from sklearn.inspection import permutation_importance
best_a = trained[trained["group"].isin(["A", "A+"])].iloc[0]["id"]
pipe = joblib.load(core.CANDIDATES_DIR / best_a / "model.joblib")
feats_a = json.loads((core.CANDIDATES_DIR / best_a / "meta.json").read_text())["features"]
te = data_cal[data_cal["split"] == "test"].sample(3000, random_state=0)
pi = permutation_importance(pipe, te[feats_a].to_numpy(np.float32), te["label"], scoring="f1", n_repeats=3, random_state=0, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=feats_a).sort_values().tail(15)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
imp.plot.barh(ax=axes[0], color="#2563eb"); axes[0].set_title(f"Permutation importance: {results[best_a]['name']} (F1 drop)")
for k in [r["id"] for r in REGISTRY if r["group"] == "C" and r["id"] in results]:
    h = pd.DataFrame(results[k].get("history", []))
    if len(h):
        axes[1].plot(h["epoch"], h["val_f1"], marker="o", label=results[k]["name"])
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation F1"); axes[1].set_title("CNN training curves"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
best_c = trained[trained["group"] == "C"].iloc[0]["id"] if (trained["group"] == "C").any() else None
if best_c:
    cnn = core.load_model(core.CANDIDATES_DIR / best_c)
    cnn_heavy = cnn.arch.startswith("deit")
    pick = pd.concat([data[(data["split"] == "test") & (data["label"] == l)].sample(3, random_state=1) for l in (0, 1)])
    fig, axes = plt.subplots(2, 6, figsize=(16, 5.6))
    for i, (_, r) in enumerate(pick.iterrows()):
        crop = cv2.imread(r["crop_path"])
        p = cnn.predict_frame(None, crop)
        cam = cnn.gradcam(crop) if not cnn_heavy else crop
        for row, im in enumerate([crop, cam]):
            ax = axes[row, i]; ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        axes[0, i].set_title(f"true: {'drowsy' if r['label'] else 'alert'}\np(drowsy) = {p:.2f}", fontsize=9)
    plt.suptitle(f"Grad-CAM of {results[best_c]['name']} on unseen participants (bottom row: where it looked)", y=1.0)
    plt.tight_layout(); plt.show()

## 7. Selecting the model for the app
**Rule** (so the choice can be explained):
1. Keep only models that run in real time: model latency ≤ 30 ms per frame on CPU.
2. Of those, pick the highest **F1 on the unseen test participants**.
3. If two are within 0.005 F1 of each other, pick the faster one.

The winner is copied to `models/best/`. **The Streamlit app loads only that folder.**

In [ ]:
deploy = trained[trained["kind"].isin(["classic_frame", "cnn_face", "fusion", "sequence", "classic_window"])]
fast = deploy[deploy["latency_ms"] <= core.MAX_MODEL_LATENCY_MS]
pool = fast if len(fast) else deploy
tied = pool[pool["f1"] >= pool["f1"].max() - core.F1_TIE_TOLERANCE]
winner = tied.sort_values("latency_ms").iloc[0]
reason = (f"Highest F1 ({winner['f1']:.1%}) among {len(pool)} real-time models (≤ {core.MAX_MODEL_LATENCY_MS:.0f} ms/frame)"
          + (f"; fastest of {len(tied)} models tied within {core.F1_TIE_TOLERANCE}." if len(tied) > 1 else "."))
shutil.rmtree(core.BEST_DIR, ignore_errors=True)
shutil.copytree(core.CANDIDATES_DIR / winner["id"], core.BEST_DIR)
wres = results[winner["id"]]
best_cfg = {"id": winner["id"], "name": wres["name"], "group": winner["group"], "kind": wres["kind"], "reason": reason,
            "metrics": {k: wres["metrics"][k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}, "latency_ms": wres["latency_ms"]}
(core.BEST_DIR / "config.json").write_text(json.dumps(best_cfg, indent=2, default=float))
paper_best = trained[trained["group"] == "paper"].iloc[0]
leader = {"groups": GROUP_TITLES, "best": best_cfg, "max_latency_ms": core.MAX_MODEL_LATENCY_MS,
          "models": [{**{k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in r.items()},
                      "confusion_matrix": results.get(r["id"], {}).get("metrics", {}).get("confusion_matrix"),
                      "roc": results.get(r["id"], {}).get("metrics", {}).get("roc")} for r in board.to_dict("records")]}
(core.REPORTS_DIR / "leaderboard.json").write_text(json.dumps(leader, indent=2, default=float))
trained.to_csv(core.REPORTS_DIR / "leaderboard.csv", index=False)
print(f"SELECTED: {wres['name']} ({winner['id']})\n  {reason}")
print(f"  accuracy {winner['accuracy']:.1%} · F1 {winner['f1']:.1%} · AUC {winner['roc_auc']:.3f} · {winner['latency_ms']:.2f} ms")
print(f"  vs best paper baseline ({paper_best['model']}): accuracy {winner['accuracy'] - paper_best['accuracy']:+.1%}, "
      f"F1 {winner['f1'] - paper_best['f1']:+.1%} on the same unseen participants")

## 8. Inference engine check
Load the engine (it reads only `models/best/`) and test it on photos of unseen participants and on a short clip that switches from alert to drowsy. In photo mode, one of the person's calibration frames serves as the optional **reference photo** (personal calibration).

In [ ]:
engine = core.DrowsinessEngine()
print("engine model:", engine.info["name"])
pick = pd.concat([data[(data["split"] == "test") & (data["label"] == l)].sample(4, random_state=7) for l in (0, 1)])
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for ax, (_, r) in zip(axes.ravel(), pick.iterrows()):
    ref = data[(data["subject_id"] == r["subject_id"]) & data["calib"]]["path"].iloc[0]
    res = engine.analyze_image(cv2.imread(r["path"]), explain=False, reference_bgr=cv2.imread(ref))
    ax.imshow(cv2.cvtColor(core.draw_overlay(cv2.imread(r["path"]), res["overlay"]), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ok = (res["state"] != "alert") == bool(r["label"])
    ax.set_title(f"true: {'drowsy' if r['label'] else 'alert'} → {res['state']}\nscore {res['score']:.0%} · KSS {res['kss']}",
                 color="green" if ok else "red", fontsize=10)
plt.suptitle("Photo mode on unseen participants, with a reference photo for calibration (green = correct)"); plt.tight_layout(); plt.show()

In [ ]:
import tempfile
subj = split["test"][0]
seq_a = data[(data["subject_id"] == subj) & (data["label"] == 0)].sort_values("order")["path"].head(150).tolist()
seq_d = data[(data["subject_id"] == subj) & (data["label"] == 1)].sort_values("order")["path"].head(150).tolist()
clip = Path(tempfile.gettempdir()) / "drowsynet_demo.mp4"
vw = cv2.VideoWriter(str(clip), cv2.VideoWriter_fourcc(*"mp4v"), 10, (227, 227))
for p in seq_a + seq_d:
    vw.write(cv2.resize(cv2.imread(p), (227, 227)))
vw.release()
vres = engine.analyze_video(clip)
tl = pd.DataFrame(vres["timeline"])
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(tl["t"], tl["score"], lw=2, label="drowsiness score", color="#4f46e5")
ax.plot(tl["t"], tl["model_prob"], lw=1, alpha=0.5, label="model probability", color="#a855f7")
ax.plot(tl["t"], tl["perclos"], lw=1.2, label="PERCLOS", color="#0891b2")
ax.axvspan(len(seq_a) / 10, tl["t"].max(), color="#fee2e2", alpha=0.6, label="ground truth: drowsy frames")
ax.axhline(core.DROWSY_MIN, ls="--", color="#dc2626"); ax.axhline(core.ALERT_MAX, ls="--", color="#d97706")
ax.set_ylim(0, 1); ax.set_xlabel("time (s)"); ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"Video mode on unseen participant {subj}: alert frames, then drowsy frames")
plt.tight_layout(); plt.show()
print({k: vres[k] for k in ("overall_state", "kss", "drowsy_share", "events")})

## 9. Streamlit app
The cell below writes `app.py`. It imports `drowsy_core.py` and the selected model from `models/best/`. Pages:
* **Live webcam**: real-time analysis with eye and mouth overlay, score, PERCLOS / blinks / yawns, personal calibration and an audio alarm. A snapshot mode also works when the camera isn't attached to the machine running the app.
* **Upload video**: timeline of the score, drowsy episodes, event counts and a CSV report.
* **Upload photo**: snapshot check with reasons and a Grad-CAM heatmap.
* **Model comparison**: leaderboard, accuracy vs speed, confusion matrices and ROC curves for every model, plus why the winner was picked.
* **How it works**: the pipeline and the comparison with the reference paper.

In [ ]:
%%writefile app.py
"""DrowsyNet Streamlit app (generated by DrowsyNet.ipynb). Run:  streamlit run app.py"""
import base64, io, json, os, tempfile, time, wave

import cv2
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

import drowsy_core as core

st.set_page_config(page_title="DrowsyNet · Driver Drowsiness Detection", page_icon="😴", layout="wide")
STATE = {"alert": ("Alert", "#16a34a"), "low_vigilance": ("Low vigilance", "#d97706"), "drowsy": ("Drowsy", "#dc2626")}
GROUP_COLORS = {"paper": "#9ca3af", "A": "#2563eb", "A+": "#ea580c", "B": "#0891b2", "C": "#7c3aed", "D": "#db2777", "E": "#059669"}

st.markdown("""<style>
.block-container{padding-top:1.6rem}
.badge{display:inline-block;padding:.35rem .9rem;border-radius:999px;color:#fff;font-weight:700;font-size:1.05rem}
.alarm{background:#dc2626;color:#fff;padding:.7rem 1rem;border-radius:.8rem;font-weight:800;text-align:center;
       font-size:1.2rem;animation:blink 1s infinite}
@keyframes blink{50%{opacity:.55}}
.bar{height:14px;border-radius:7px;background:#e2e8f0;overflow:hidden}
.bar>div{height:100%;border-radius:7px}
</style>""", unsafe_allow_html=True)


@st.cache_resource(show_spinner="Loading face landmarker and the selected model…")
def get_engine():
    return core.DrowsinessEngine()


@st.cache_data
def beep_b64():
    sr, t = 22050, np.linspace(0, 0.35, int(22050 * 0.35), False)
    tone = (np.sign(np.sin(2 * np.pi * 880 * t)) * 0.25 * np.exp(-4 * t) * 32767).astype(np.int16)
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(tone.tobytes())
    return base64.b64encode(buf.getvalue()).decode()


def badge(state):
    label, color = STATE[state]
    return f'<span class="badge" style="background:{color}">{label}</span>'


def score_bar(score):
    color = STATE[core.state_of(score)][1]
    return f'<div class="bar"><div style="width:{score * 100:.0f}%;background:{color}"></div></div>'


def gauge(score, title="Drowsiness score", height=230):
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=score * 100, number={"suffix": "%"}, title={"text": title},
        gauge={"axis": {"range": [0, 100]}, "bar": {"color": STATE[core.state_of(score)][1]},
               "steps": [{"range": [0, 40], "color": "#dcfce7"}, {"range": [40, 65], "color": "#fef3c7"},
                         {"range": [65, 100], "color": "#fee2e2"}]}))
    fig.update_layout(height=height, margin=dict(l=20, r=20, t=50, b=10))
    return fig


def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def annotate(frame, r):
    color = (68, 68, 239) if r.get("state") == "drowsy" else (238, 211, 34)
    vis = core.draw_overlay(frame, r.get("overlay"), color=color)
    label, hexc = STATE[r["state"]]
    bgr = tuple(int(hexc[i:i + 2], 16) for i in (5, 3, 1))
    cv2.rectangle(vis, (0, 0), (vis.shape[1], 34), (0, 0, 0), -1)
    cv2.putText(vis, f"{label}  {r['score'] * 100:.0f}%  KSS {r['kss']}", (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, bgr, 2, cv2.LINE_AA)
    if not r["face"]:
        cv2.putText(vis, "no face", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    return vis


engine = get_engine()

# ------------------------------------------------------------------------- sidebar
with st.sidebar:
    st.title("😴 DrowsyNet")
    st.caption("Driver drowsiness detection")
    page = st.radio("Page", ["Live webcam", "Upload video", "Upload photo", "Model comparison", "How it works"], label_visibility="collapsed")
    st.divider()
    st.markdown("**Model in use**")
    st.write(engine.info["name"])
    if "metrics" in engine.info:
        m = engine.info["metrics"]
        c1, c2 = st.columns(2)
        c1.metric("Accuracy", f"{m['accuracy']:.1%}")
        c2.metric("F1", f"{m['f1']:.1%}")
        st.caption(engine.info.get("reason", ""))
    st.divider()
    sound = st.toggle("Alarm sound", value=True)

# ------------------------------------------------------------------------- live webcam
if page == "Live webcam":
    st.header("Live webcam")
    mode = st.radio("Mode", ["Live stream (camera on this computer)", "Snapshot (browser camera)"], horizontal=True)
    if mode.startswith("Snapshot"):
        shot = st.camera_input("Take a picture")
        if shot:
            img = cv2.imdecode(np.frombuffer(shot.getvalue(), np.uint8), cv2.IMREAD_COLOR)
            r = engine.analyze_image(img, explain=False)
            if not r["face"]:
                st.error("No face detected.")
            else:
                c1, c2 = st.columns([3, 2])
                c1.image(bgr2rgb(core.draw_overlay(img, r["overlay"])), use_container_width=True)
                c2.plotly_chart(gauge(r["score"]), use_container_width=True)
                c2.markdown(badge(r["state"]) + f" &nbsp; KSS {r['kss']}/9 · {r['kss_label']}", unsafe_allow_html=True)
                for x in r["reasons"] or ["No drowsiness signs detected."]:
                    c2.write("• " + x)
    else:
        c0, c1, c2 = st.columns([1, 1, 2])
        cam_index = c0.number_input("Camera index", 0, 5, 0)
        run = c1.toggle("▶ Start camera", key="cam_on")
        c2.caption("The first ~20 s calibrate to your face: look at the road (screen) normally. Use the toggle to stop. "
                   "Any button press restarts calibration.")
        left, right = st.columns([3, 2])
        frame_ph, alarm_ph, audio_ph = left.empty(), left.empty(), left.empty()
        with right:
            state_ph, bar_ph, kss_ph, calib_ph = st.empty(), st.empty(), st.empty(), st.empty()
            m1, m2 = st.columns(2)
            perclos_ph, blink_ph = m1.empty(), m2.empty()
            m3, m4 = st.columns(2)
            yawn_ph, closure_ph = m3.empty(), m4.empty()
            chart_ph, reasons_ph, fps_ph = st.empty(), st.empty(), st.empty()
        if run:
            cap = cv2.VideoCapture(int(cam_index), cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY)
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            if not cap.isOpened():
                st.error("Could not open the camera. Try another camera index or use Snapshot mode.")
            else:
                session, t0, last, last_beep, hist, n = engine.new_session(), time.time(), 0.0, 0.0, [], 0
                try:
                    while True:
                        ok, frame = cap.read()
                        if not ok:
                            st.error("Camera stopped sending frames."); break
                        now = time.time()
                        if now - last < 1.0 / core.TARGET_FPS:
                            continue
                        last = now
                        frame = cv2.flip(frame, 1)
                        r = engine.analyze_stream_frame(session, frame, now - t0)
                        tm = r["temporal"]
                        frame_ph.image(bgr2rgb(annotate(frame, r)), use_container_width=True)
                        alarm_ph.markdown('<div class="alarm">⚠ WAKE UP: drowsiness detected</div>' if r["alarm"] else "", unsafe_allow_html=True)
                        if r["alarm"] and sound and now - last_beep > 1.5:
                            last_beep = now
                            audio_ph.markdown(f'<audio autoplay src="data:audio/wav;base64,{beep_b64()}"></audio><!--{now}-->', unsafe_allow_html=True)
                        state_ph.markdown(badge(r["state"]) + f" &nbsp; <b style='font-size:1.6rem'>{r['score'] * 100:.0f}%</b>", unsafe_allow_html=True)
                        bar_ph.markdown(score_bar(r["score"]), unsafe_allow_html=True)
                        kss_ph.caption(f"KSS {r['kss']}/9 · {r['kss_label']}")
                        calib_ph.progress(tm["calibration_progress"], text="Calibrated to your face ✓" if tm["calibrated"] and tm["model_calibrated"] else "Calibrating to your face… stay alert and look ahead")
                        perclos_ph.metric("PERCLOS (30 s)", f"{tm['perclos']:.0%}")
                        blink_ph.metric("Blinks / min", f"{tm['blinks_per_min']:.0f}")
                        yawn_ph.metric("Yawns (1 min)", tm["yawns"])
                        closure_ph.metric("Eyes closed now", f"{tm['current_closure_s']:.1f} s")
                        hist.append({"t": round(now - t0, 1), "score": r["score"]})
                        n += 1
                        if n % 10 == 0:
                            chart_ph.line_chart(pd.DataFrame(hist[-300:]).set_index("t"), height=150)
                            reasons_ph.markdown("\n".join(f"- {x}" for x in r["reasons"]) or "No drowsiness signs detected.")
                            fps_ph.caption(f"{r['latency_ms']:.0f} ms per frame")
                finally:
                    cap.release()

# ------------------------------------------------------------------------- video
elif page == "Upload video":
    st.header("Upload a driving video")
    up = st.file_uploader("Video file", type=["mp4", "mov", "avi", "mkv", "m4v"])
    if up and st.button("Analyse video", type="primary"):
        with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(up.name)[1]) as f:
            f.write(up.getbuffer())
        bar = st.progress(0.0, text="Analysing ~10 frames per second…")
        try:
            st.session_state["video_res"] = engine.analyze_video(f.name, progress=lambda p: bar.progress(p, text=f"Analysing… {p:.0%}"))
            st.session_state["video_name"] = up.name
        except ValueError as e:
            st.error(str(e))
        finally:
            bar.empty(); os.unlink(f.name)
    res = st.session_state.get("video_res")
    if res:
        c1, c2 = st.columns([1, 2])
        c1.plotly_chart(gauge(res["overall_score"], "Overall (90th percentile)"), use_container_width=True)
        c1.markdown(badge(res["overall_state"]) + f" &nbsp; KSS {res['kss']}/9 · {res['kss_label']}", unsafe_allow_html=True)
        cols = c2.columns(4)
        for col, (k, v) in zip(cols * 2, [("Duration", f"{res['duration_s']:.0f} s"), ("Time drowsy", f"{res['drowsy_share']:.0%}"),
                                          ("Peak score", f"{res['peak_score']:.0%}"), ("Face visible", f"{res['face_ratio']:.0%}"),
                                          ("Blinks", res["events"]["blinks"]), ("Long closures", res["events"]["long_closures"]),
                                          ("Yawns", res["events"]["yawns"]), ("Head nods", res["events"]["nods"])]):
            col.metric(k, v)
        tl = pd.DataFrame(res["timeline"])
        fig = go.Figure()
        for ep in res["episodes"]:
            fig.add_vrect(x0=ep["start"], x1=ep["end"], fillcolor="#fee2e2", opacity=0.7, line_width=0)
        fig.add_scatter(x=tl["t"], y=tl["score"], name="Drowsiness score", line=dict(color="#4f46e5", width=3))
        fig.add_scatter(x=tl["t"], y=tl["perclos"], name="PERCLOS", line=dict(color="#0891b2"))
        fig.add_scatter(x=tl["t"], y=tl["model_prob"], name="Model probability", line=dict(color="#a855f7", width=1), opacity=0.6)
        fig.add_hline(y=core.DROWSY_MIN, line_dash="dash", line_color="#dc2626")
        fig.add_hline(y=core.ALERT_MAX, line_dash="dash", line_color="#d97706")
        fig.update_layout(height=360, yaxis_range=[0, 1], xaxis_title="time (s)", margin=dict(t=30, b=10), legend=dict(orientation="h"))
        st.plotly_chart(fig, use_container_width=True)
        if res["episodes"]:
            st.write("**Drowsy episodes:** " + ", ".join(f"{e['start']:.1f}–{e['end']:.1f} s" for e in res["episodes"]))
        st.download_button("Download CSV report", tl.to_csv(index=False), f"drowsiness_{st.session_state['video_name']}.csv", "text/csv")

# ------------------------------------------------------------------------- photo
elif page == "Upload photo":
    st.header("Upload a photo")
    st.info("A single photo has no time information, so this is a **snapshot check** (eye state, yawning, head pose). Use video or webcam for the full analysis.")
    c_up, c_ref = st.columns(2)
    up = c_up.file_uploader("Face photo to check", type=["jpg", "jpeg", "png", "bmp", "webp"])
    ref_up = c_ref.file_uploader("Optional: reference photo of the same person while alert (personal calibration)",
                                 type=["jpg", "jpeg", "png", "bmp", "webp"], disabled=not engine.needs_calibration,
                                 help="The selected model compares the face with this person's own alert face. Without it, an average person is used.")
    if up:
        img = cv2.imdecode(np.frombuffer(up.getvalue(), np.uint8), cv2.IMREAD_COLOR)
        ref = cv2.imdecode(np.frombuffer(ref_up.getvalue(), np.uint8), cv2.IMREAD_COLOR) if ref_up else None
        r = engine.analyze_image(img, reference_bgr=ref)
        if not r["face"]:
            st.error("No face detected in this image.")
        else:
            c1, c2 = st.columns([3, 2])
            c1.image(bgr2rgb(core.draw_overlay(img, r["overlay"])), use_container_width=True)
            with c2:
                st.plotly_chart(gauge(r["score"]), use_container_width=True)
                st.markdown(badge(r["state"]) + f" &nbsp; KSS {r['kss']}/9 · {r['kss_label']}", unsafe_allow_html=True)
                if engine.needs_calibration:
                    st.caption("Personal calibration: " + ("on (reference photo)" if r["calibrated"] else "off (average-person baseline)"))
                s = r["signals"]
                a, b = st.columns(2)
                a.metric("Model probability", f"{r['model_prob']:.0%}")
                b.metric("Eye closure", f"{s['eye_closure']:.0%}", help=f"EAR {s['ear']}")
                a.metric("Jaw open", f"{s['jaw_open']:.0%}", help=f"MAR {s['mar']}")
                b.metric("Head pitch / yaw", f"{s['pitch']:.0f}° / {s['yaw']:.0f}°")
                st.markdown("**Why**")
                for x in r["reasons"] or ["No drowsiness signs detected."]:
                    st.write("• " + x)
            imgs = [("Face crop seen by the model", r["face_crop"])] + ([("Grad-CAM: red = pushed towards drowsy", r["gradcam"])] if r.get("gradcam") is not None else [])
            for col, (cap, im) in zip(st.columns(4), imgs):
                col.image(bgr2rgb(im), caption=cap, use_container_width=True)

# ------------------------------------------------------------------------- comparison
elif page == "Model comparison":
    st.header("Model comparison")
    p = core.REPORTS_DIR / "leaderboard.json"
    if not p.exists():
        st.warning("Run the notebook first to train and compare the models.")
        st.stop()
    lb = json.loads(p.read_text())
    df = pd.DataFrame(lb["models"])
    tr = df[df["status"] == "trained"].copy()
    best = lb["best"]
    paper = tr[tr["group"] == "paper"].sort_values("f1").iloc[-1]
    with st.container(border=True):
        st.markdown(f"### 🏆 Model used by the app: {best['name']}")
        st.caption(best["reason"])
        k = st.columns(5)
        k[0].metric("Accuracy", f"{best['metrics']['accuracy']:.1%}")
        k[1].metric("F1", f"{best['metrics']['f1']:.1%}")
        k[2].metric("ROC-AUC", f"{best['metrics']['roc_auc']:.3f}")
        k[3].metric("Latency", f"{best['latency_ms']:.2f} ms")
        k[4].metric("vs paper baseline (F1)", f"{(best['metrics']['f1'] - paper['f1']) * 100:+.1f} pts")
    metric = st.radio("Metric", ["accuracy", "f1", "roc_auc", "precision", "recall"], horizontal=True,
                      format_func=lambda m: {"roc_auc": "ROC-AUC"}.get(m, m.capitalize()))
    t = tr.sort_values(metric)
    fig = go.Figure(go.Bar(x=t[metric], y=t["model"], orientation="h", marker_color=[GROUP_COLORS[g] for g in t["group"]],
                           marker_line=dict(width=[3 if i == best["id"] else 0 for i in t["id"]], color="#111827"),
                           text=[f"{v:.1%}" if metric != "roc_auc" else f"{v:.3f}" for v in t[metric]], textposition="outside"))
    fig.update_layout(height=max(380, 26 * len(t)), xaxis_range=[0, 1.1], margin=dict(l=10, r=10, t=10, b=10))
    st.plotly_chart(fig, use_container_width=True)
    st.caption(" · ".join(f"<span style='color:{GROUP_COLORS[g]}'>■</span> {title}" for g, title in lb["groups"].items()), unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    fig = go.Figure()
    for g, sub in tr.groupby("group"):
        fig.add_scatter(x=sub["latency_ms"], y=sub["f1"], mode="markers", name=lb["groups"][g], text=sub["model"],
                        marker=dict(size=12, color=GROUP_COLORS[g]), hovertemplate="%{text}<br>F1 %{y:.1%}<br>%{x:.2f} ms")
    fig.add_vline(x=lb["max_latency_ms"], line_dash="dash", line_color="#ef4444", annotation_text="real-time limit")
    fig.update_layout(title="Accuracy vs speed", xaxis_type="log", xaxis_title="latency per frame (ms, CPU)", yaxis_title="F1",
                      height=420, legend=dict(orientation="h", y=-0.25))
    c1.plotly_chart(fig, use_container_width=True)
    with c2:
        sel = st.selectbox("Model details", tr.sort_values("f1", ascending=False)["model"].tolist())
        row = tr[tr["model"] == sel].iloc[0]
        cm = np.array(row["confusion_matrix"])
        a, b = st.columns(2)
        hm = go.Figure(go.Heatmap(z=cm, x=["pred alert", "pred drowsy"], y=["alert", "drowsy"], text=cm, texttemplate="%{text}",
                                  colorscale="Blues", showscale=False))
        hm.update_layout(height=260, margin=dict(t=30, b=10), title="Confusion matrix", yaxis_autorange="reversed")
        a.plotly_chart(hm, use_container_width=True)
        roc = go.Figure(go.Scatter(x=row["roc"]["fpr"], y=row["roc"]["tpr"], line=dict(color=GROUP_COLORS[row["group"]], width=3)))
        roc.add_scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash", color="#cbd5e1"))
        roc.update_layout(height=260, margin=dict(t=30, b=10), title=f"ROC (AUC {row['roc_auc']:.3f})", showlegend=False)
        b.plotly_chart(roc, use_container_width=True)
    show = tr.sort_values("f1", ascending=False)[["model", "group", "accuracy", "precision", "recall", "f1", "roc_auc", "cv_f1", "latency_ms", "size_mb", "budget"]]
    st.dataframe(show.style.format({c: "{:.1%}" for c in ["accuracy", "precision", "recall", "f1", "cv_f1"]} |
                                   {"roc_auc": "{:.3f}", "latency_ms": "{:.2f}", "size_mb": "{:.1f}"}, na_rep="–"),
                 use_container_width=True, hide_index=True)
    pend = df[df["status"] != "trained"]
    if len(pend):
        st.caption(f"Not trained ({len(pend)}): " + ", ".join(pend["model"]) + ". Groups B and D need video data (UTA-RLDD).")

# ------------------------------------------------------------------------- about
else:
    st.header("How it works")
    st.markdown("""
**Pipeline**: quality gate (blur / exposure, CLAHE for dark frames) → MediaPipe Face Landmarker (478 points, 52 blendshapes,
head pose) → per-frame features and face crop → **selected model** → temporal layer (personal calibration, PERCLOS, blinks,
long closures, yawns, nods) → smoothed score → alert / low vigilance / drowsy + KSS level.

| Aspect | Reference paper (TENCON 2019) | DrowsyNet |
|---|---|---|
| Landmarks | dlib, 68 points | MediaPipe, 478 points + blendshapes + head pose |
| Personal calibration | average over the whole recording | z-scores against the driver's first ~20 s (alert) |
| Time | single frames | PERCLOS, blink, yawn, nod events |
| Evaluation | same person in training and testing | unseen test participants + 5-fold group CV |
| Models | 4 classifiers | paper baseline, 14 classic ML (raw and calibrated), 8 CNN/ViT, stacking, late fusion (+ temporal models on video) |
| Robustness | blurred frames broke 3 of 10 subjects | quality gate, low-light enhancement, CNN branch |
| Score | product of averages, unbounded | bounded 0–1 score mapped to KSS 1–9 |
| Explainability | none | reasons + Grad-CAM |

*Research prototype, not a certified driver-safety device.*
""")

### 9.1 Launch the app
Runs Streamlit in the background and prints the URL. Open **http://localhost:8501** in a browser. On Google Colab the app needs a tunnel (for example `npx localtunnel --port 8501`); running locally is recommended so the live webcam works.

In [ ]:
import subprocess, socket
def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0
if port_open(8501):
    print("Streamlit is already running → http://localhost:8501")
else:
    app_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                                cwd=PROJECT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if port_open(8501):
            break
        time.sleep(1)
    print("DrowsyNet app running → http://localhost:8501   (run the next cell to stop it)")

In [ ]:
# Stop the app
try:
    app_proc.terminate(); print("stopped")
except NameError:
    print("the app was not started from this notebook session")

## 10. Conclusion
* Each model is scored on participants it **never saw**, so the numbers estimate how the system works for a new driver. The reference paper trained and tested on the same person, which hides this generalisation gap.
* **Personal calibration is the biggest single gain.** Raw Group A vs Group A+ shows that comparing a face with the same person's own alert baseline beats any choice of classifier, because differences between people dominate DDD. Richer landmarks (478 points + blendshapes + head pose) beat the paper's EAR/MAR-only features on the same split (see the selection cell for the exact gain). The benchmark of classic ML, CNNs / ViT and ensembles shows which model families suit the task, and the speed–accuracy chart shows why the deployed model was chosen.
* In video and webcam mode, the temporal layer (personal calibration, PERCLOS, long closures, yawns, nods) turns noisy per-frame predictions into a stable score with an alarm.

**Limitations.** DDD has only 28 participants, so the test set is a few people and scores change with the split (see the CV spread). DDD labels are per video segment, not per frame, so some "drowsy" frames show open eyes. A single photo can only give a snapshot. CNNs were trained on a reduced CPU budget.

**Future work.** Train Groups B and D on UTA-RLDD (code ready: set `USE_RLDD = True`), train the CNNs on a GPU with full data, add infrared / night-time data, and personalise further with a few minutes of a driver's own data.

**References**
1. C. Yashwanth, J. S. Kirar. *Driver's Drowsiness Detection.* IEEE TENCON 2019.
2. T. Soukupová, J. Čech. *Real-Time Eye Blink Detection using Facial Landmarks.* CVWW 2016.
3. R. Ghoddoosian, M. Galib, V. Athitsos. *A Realistic Dataset and Baseline Temporal Model for Early Drowsiness Detection.* CVPRW 2019.
4. Google MediaPipe Face Landmarker. https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker
5. Driver Drowsiness Dataset (DDD), Kaggle. https://www.kaggle.com/datasets/ismailnasri20/driver-drowsiness-dataset-ddd